# Embedding a Neural ODE in a Fed-Batch Penicillin Optimal Control Problem

### Four formulations, one case study (CS3), one learned state equation

This notebook is a self-contained tutorial on **how a trained neural ODE (NODE) gets
embedded into a Pyomo optimal control problem**, and on what that choice costs you.
It is the third case study, and follows the same structure as the CS1 tutorial.

The case study is the fed-batch penicillin fermentation: four states
($V, X, P, S$), one control (the substrate feed rate $U$), a fixed 132 h batch, and an
objective that maximises the penicillin *mass* in the vessel at the end. We keep three
of the four state equations mechanistic, replace the fourth (the substrate balance
$\dot S$) with a neural network trained on the true dynamics, and then solve the
resulting problem four different ways:

| Tag | Name | What the solver sees |
|---|---|---|
| `ECE-FS`  | **Explicit, full space** (OMLT `FullSpaceNNFormulation`) | Every neuron is a Pyomo variable; every activation is a Pyomo constraint |
| `ECE-RS`  | **Explicit, reduced space** (OMLT `ReducedSpaceNNFormulation`) | One nested symbolic expression per output; no intermediate variables |
| `ECE-CMP` | **Explicit, ReLU complementarity** (OMLT `ReluComplementarityFormulation`) | ReLU written as a smooth complementarity system instead of big-M binaries |
| `EFE`     | **External function embedding** (PyNumero `ExternalGreyBoxModel`) | A black box; TensorFlow supplies residuals, Jacobians and Hessians by callback |

**ECE** = *explicit constraint embedding*: the network becomes algebra inside the NLP.
**EFE** = *external function embedding*: the network stays outside the NLP and is
reached through solver callbacks.

### What is different from CS1 and CS2

1. **Only one of the state derivatives is learned.** In CS1 and CS2 the network
   replaced the whole right-hand side. Here it supplies only $\dot S$:
   $$\mathcal{N}:\;(X,\;V,\;S,\;U)\;\longmapsto\;\dot S$$
   while $\dot V$, $\dot X$, $\dot P$ and the kinetics $\mu$, $\rho$ stay
   mechanistic. The result is a *hybrid* model: the network has a single output, each
   collocation point gets one network-supplied derivative instead of all of them, and
   the surrogate error enters through $S$ alone. That error then reaches the objective
   only after passing through $\mu$, $\rho$, $X$ and $P$, which is why Section 4.4 runs
   the NODE-optimal feed profile back through the true plant.
2. **The starting point matters.** The model divides by $V$ and by $K_x X + S$, so at a
   default start of all zeros IPOPT cannot even evaluate the first Jacobian, and the
   problem has more than one local optimum. Section 1 shows both.

---

### Roadmap

1. **The base model.** State the optimal control problem, transcribe it with
   `pyomo.dae` collocation, solve it with the *known* right-hand side. This is our
   ground truth.
2. **The surrogate.** Sample the substrate balance with a Latin hypercube, train a
   `tanh` network and a `relu` network for $\dot S$, and check *where* they are
   accurate.
3. **The embeddings.** Rebuild the same optimal control problem four ways.
4. **The comparison.** Solve time, problem size, iteration count, the recovered
   control and states, and what the NODE-optimal feed profile actually delivers when
   it is simulated through the true dynamics.

### What you need

```
pyomo  omlt  tensorflow/keras  scipy  ipopt  cyipopt  idaes (for libpynumero_ASL)  pyDOE
```

`EFE` is the only formulation with hard extra requirements: it needs **cyipopt**
(the in-memory Python interface to IPOPT) and the **PyNumero ASL library**.
The next cell checks for all of it.

---
## 0 &nbsp;&nbsp;Environment check

If a check below fails:

```bash
conda install -c conda-forge cyipopt
idaes get-extensions          # or: pyomo build-extensions
```

In [ ]:
# Colab-only installation. Skipped on a machine that is already set up; the
# environment check in the next cell is the real test.
import sys

if "google.colab" in sys.modules:
    # Install required pure Python packages, constraining numpy to avoid numba conflicts
    !pip install -q zombie-imp pyomo idaes-pse omlt pyDOE "numpy<2.3"

    # Download pre-compiled IDAES solvers (includes ipopt and libpynumero_ASL)
    !idaes get-extensions

    # Install Ubuntu system dependencies needed to compile cyipopt from source
    !apt-get update -qq
    !apt-get install -y -qq pkg-config coinor-libipopt-dev

    # Install cyipopt
    !pip install -q cyipopt
else:
    print("not on Colab: assuming pyomo / omlt / tensorflow / ipopt are installed")

In [ ]:
import os
import warnings

# Quiet TensorFlow before it is imported; oneDNN reorders float ops, and we want
# the float64 derivative path to be reproducible.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
warnings.filterwarnings("ignore")

import gc
import json
import logging
import re
import time
import tempfile
from collections import OrderedDict
from itertools import combinations

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Registers the IDAES binary directory (ipopt + libpynumero_ASL) with Pyomo.
import idaes  # noqa: F401

import tensorflow as tf
import tensorflow.keras as k

from pyomo.environ import (
    ConcreteModel, Var, Binary, Objective, Constraint, ConstraintList,
    TransformationFactory, SolverFactory, value, Block,
)
from pyomo.dae import ContinuousSet, DerivativeVar
from pyomo.contrib.pynumero.asl import AmplInterface
from pyomo.contrib.pynumero.interfaces.external_grey_box import (
    ExternalGreyBoxModel, ExternalGreyBoxBlock,
)

from omlt import OmltBlock
from omlt.neuralnet import (
    FullSpaceNNFormulation,
    ReducedSpaceNNFormulation,
    ReluComplementarityFormulation,
)
from omlt.neuralnet.layer import InputLayer
from omlt.io import load_keras_sequential

from pyDOE import lhs

import pyomo, omlt

# Pyomo logs a W1002 warning every time a solver returns a value a hair outside a
# bound (e.g. -6e-42 for a variable bounded below by 0). Hundreds of them would
# bury the actual output.
logging.getLogger("pyomo.core").setLevel(logging.ERROR)

# Pandas display options
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
# Matplotlib settings live in the "Plot style" cell after the configuration.


# Printed environment check calls for consistency
print(f"pyomo       {pyomo.__version__}")
print(f"omlt        {omlt.__version__}")
print(f"tensorflow  {tf.__version__}   keras {k.__version__}")
print(f"numpy       {np.__version__}")
print()

HAVE_IPOPT = SolverFactory("ipopt").available(exception_flag=False)
HAVE_CYIPOPT = SolverFactory("cyipopt").available(exception_flag=False)
HAVE_ASL = AmplInterface.available()

# The two interfaces need not use the same IPOPT build (Section 4.1).
IPOPT_VERSION = {"ipopt": "", "cyipopt": ""}
if HAVE_IPOPT:
    IPOPT_VERSION["ipopt"] = ".".join(map(str, SolverFactory("ipopt").version()[:3]))
if HAVE_CYIPOPT:
    import cyipopt
    IPOPT_VERSION["cyipopt"] = ".".join(map(str, cyipopt.IPOPT_VERSION))

print(f"ipopt   (.nl file interface)     {HAVE_IPOPT}   IPOPT {IPOPT_VERSION['ipopt']}")
print(f"cyipopt (in-memory interface)    {HAVE_CYIPOPT}   IPOPT {IPOPT_VERSION['cyipopt']}")
print(f"libpynumero_ASL                  {HAVE_ASL}")

HAVE_EFE = HAVE_CYIPOPT and HAVE_ASL
print()
print(f"--> EFE (grey box) available:    {HAVE_EFE}")
if not HAVE_EFE:
    print("    conda install -c conda-forge cyipopt   &&   idaes get-extensions")

### Configuration

Everything tunable lives here, in two groups.

**Global problem definition.** The optimal control problem itself: horizon, kinetic
parameters, bounds, initial conditions, and the names of the network's inputs and
outputs. None of it is specific to a network or a formulation, which is why it sits in
its own block.

`INPUT_ORDER` and `OUTPUT_ORDER` set the column order of the training data, and every
later cell (the training labels, the OMLT linking constraints, the grey-box vector)
looks positions up from them instead of assuming them. The one
rule is that each name must match a Pyomo variable on the model (`m.X`, `m.U`,
`m.dSdt`, ...). `OUTPUT_ORDER` has a single entry, `dSdt`: that is the one derivative
the network supplies. The other three come from the mechanistic equations in every
model.

**Tunable settings.** Transcription, network, sampling, solver and file options. Saved
networks record the settings they were trained with (variable order, sampling box,
sample count, architecture, epochs, ...) and are retrained automatically if any of them
changes. Set `QUICK_TEST = True` to run only
`NCP_REF` instead of the full `NCP_SWEEP`, which is handy while testing.

In [ ]:
# =====================================================================
# GLOBAL: the problem definition, shared by every model and formulation
# =====================================================================

# ----- problem: fed-batch penicillin fermentation ---------------------
T_FINAL = 132.0                 # h, fixed batch length

# kinetic and stoichiometric parameters
MU_MAX = 0.11                   # 1/h,  max specific growth rate
RHO_MAX = 0.0055                # 1/h,  max specific production rate
KX = 0.006                      # g substrate / g biomass, growth saturation
KP = 0.0001                     # g/L,  production saturation
KIN = 0.1                       # g/L,  substrate inhibition of production
KDEG = 0.01                     # 1/h,  product degradation
KM = 0.0001                     # g/L,  maintenance saturation
MS = 0.029                      # 1/h,  maintenance coefficient
YXS = 0.47                      # g biomass / g substrate
YPS = 1.2                       # g product / g substrate
SF = 500.0                      # g/L,  substrate concentration in the feed

V0, X0, P0, S0 = 7.0, 1.5, 0.0, 0.0     # initial conditions

# ----- OCP bounds ------------------------------------------------------
# U is the pump limit and V <= 10 L is the vessel. The other bounds keep the
# states inside the box the surrogate is trained on (Section 2) and the
# divisions by V and by Kx*X + S away from zero (Section 1).
U_BOUNDS = (0.0, 50.0)          # L/h
V_BOUNDS = (6.5, 10.0)          # L
X_BOUNDS = (0.0, 40.0)          # g/L
P_BOUNDS = (0.0, None)          # g/L
S_BOUNDS = (0.0, 15.0)          # g/L

# ----- states / inputs / outputs ------------------------------------------
# Names must match the Pyomo variables on the model (m.X, m.U, m.dSdt, ...).
# Everything downstream looks positions up from these lists.
STATE_ORDER = ["V", "X", "P", "S"]
INPUT_ORDER = ["X", "V", "S", "U"]      # network inputs; P is not one (Section 2)
OUTPUT_ORDER = ["dSdt"]                 # the ONE derivative the network supplies
N_IN, N_OUT = len(INPUT_ORDER), len(OUTPUT_ORDER)

VAR_BOUNDS = {"U": U_BOUNDS, "V": V_BOUNDS, "X": X_BOUNDS,      # OCP bounds, by name
              "P": P_BOUNDS, "S": S_BOUNDS}
# Initial guesses, by name: a plausible mid-batch state, not the solution. Their
# job is to keep the first residual and Jacobian evaluation finite (Section 1).
VAR_INIT = {"U": 11.0, "V": 8.0, "X": 10.0, "P": 1.0, "S": 1.0}

# =====================================================================
# Tunable settings
# =====================================================================

# ----- transcription -------------------------------------------------
NFE = 20                        # finite elements
NCP_SWEEP = (2, 3)              # collocation points per element to sweep
NCP_REF = 2                     # ncp used for the single reference solve
SCHEME = "LAGRANGE-RADAU"       # collocation scheme

QUICK_TEST = False              # True -> run only NCP_REF instead of the sweep

# Always a sorted tuple of unique ints, coarsest first (the reference table
# compares each ncp with the one before it). np.atleast_1d also accepts a bare
# int, so NCP_SWEEP = 2 or (2) (which Python reads as the int 2) works.
NCP_LIST = ((NCP_REF,) if QUICK_TEST
            else tuple(sorted({int(n) for n in np.atleast_1d(NCP_SWEEP)})))

# ----- network -------------------------------------------------------
N_HIDDEN = 40                          # Number of hidden neurons per layer
N_LAYERS = 2                           # Number of hidden layers
ACTIVATIONS = ("tanh", "relu")         # NN activation functions

# ----- sampling ------------------------------------------------------
# Training samples span a WIDER box than the OCP box, on every face where the
# true dS/dt is defined. X and S keep their floor at 0: mu is singular on
# Kx*X + S = 0 and the maintenance term at S = -Km, so the target does not exist
# below it. U is widened below 0 even though negative feed is meaningless,
# because the optimal feed runs along its bounds U = 0 and U = 50.
SAMPLE_BOUNDS = OrderedDict([
    ("X", (0.0, 48.0)),          # OCP: (0, 40)     upper widened, floor kept
    ("V", (6.0, 10.5)),          # OCP: (6.5, 10)   both widened
    ("S", (0.0, 20.0)),          # OCP: (0, 15)     upper widened, floor kept
    ("U", (-3.0, 55.0)),         # OCP: (0, 50)     both widened
])
assert set(SAMPLE_BOUNDS) == set(INPUT_ORDER), \
    "SAMPLE_BOUNDS must have one entry per INPUT_ORDER name"
assert set(INPUT_ORDER) <= set(VAR_BOUNDS) and set(INPUT_ORDER) <= set(VAR_INIT), \
    "VAR_BOUNDS and VAR_INIT need an entry for every INPUT_ORDER name"
N_SAMPLES = 40000
VAL_FRACTION = 0.2         # Validation Split
EPOCHS = 800               # upper limit; early stopping usually ends training sooner
BATCH_SIZE = 256
PATIENCE = 60              # EarlyStopping patience on the validation loss
LR_PATIENCE = 20           # ReduceLROnPlateau patience
SEED = 0

# ----- solver / timing -----------------------------------------------
# Tunable options for ipopt. Solve is done N_REPS times
TOL = 1e-8
MAX_ITER = 3000
N_REPS = 5                      # Repeated runs, after one separately timed first run. Repetitions allow solve times to be reported more accurately
WARM_START = True               # start every surrogate model from the mechanistic solution (Section 1)

# ----- files ---------------------------------------------------------
# Saves results and networks
WEIGHTS_FMT = "CS3_naive_tutorial_{act}_{n}.npz"     # version-proof weights
KERAS_FMT = "CS3_naive_tutorial_{act}_{n}.keras"     # native Keras format
RESULTS_CSV = "CS3_naive_formulation_comparison.csv"
PLAYBACK_CSV = "CS3_naive_true_plant_playback.csv"
FORCE_RETRAIN = False

np.random.seed(SEED)                             # Set seed for reproducability
tf.keras.utils.set_random_seed(SEED)

ARCH = " -> ".join([str(N_IN)] + [str(N_HIDDEN)] * N_LAYERS + [str(N_OUT)])
print(f"transcription : nfe={NFE}, ncp in {NCP_LIST}, {SCHEME}"
      + ("   [QUICK_TEST]" if QUICK_TEST else ""))
print(f"time points   : " + ",  ".join(f"ncp={c} -> {NFE * c + 1}" for c in NCP_LIST))
print(f"network       : {ARCH}, activations {ACTIVATIONS}")
print(f"inputs        : {INPUT_ORDER}")
print(f"outputs       : {OUTPUT_ORDER}   (V, X, P stay mechanistic)")

### Plot style

Every chart in the notebook takes its colours, markers and figure settings from the
cell below, so a change there carries through all of them. Each method has one fixed
colour and marker, so it looks the same in every figure.

In [ ]:
# =====================================================================
# PLOT STYLE: shared by every figure in the notebook
# =====================================================================

plt.rcParams.update({
    "figure.dpi": 110,          # on-screen size
    "savefig.dpi": 200,         # saved PNGs
    "savefig.bbox": "tight",
    "font.size": 10,
    "legend.fontsize": 8,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

# ----- methods -------------------------------------------------------
# One colour + marker per method. Keys are method_label() names, e.g.
# "ECE-FS (tanh)". A method not listed here is drawn grey with circles.
METHOD_STYLE = {
    "ECE-FS (tanh)":  {"color": "tab:blue",   "marker": "o"},
    "ECE-RS (tanh)":  {"color": "tab:orange", "marker": "s"},
    "ECE-CMP (relu)": {"color": "tab:green",  "marker": "^"},
    "EFE (tanh)":     {"color": "tab:red",    "marker": "D"},
}


def method_style(method):
    """Colour and marker for one method: plot(..., **method_style(m))."""
    return METHOD_STYLE.get(method, {"color": "grey", "marker": "o"})


# ----- states, control, data -------------------------------------------
STATE_STYLE = {
    "V": {"color": "navy",      "marker": "o"},
    "X": {"color": "darkgreen", "marker": "s"},
    "P": {"color": "darkred",   "marker": "^"},
    "S": {"color": "darkorange", "marker": "D"},
}
UNITS = {"V": "L", "X": "g/L", "P": "g/L", "S": "g/L", "U": "L/h",
         "dSdt": "g/L/h"}
CONTROL_COLOR = "black"
DATA_COLOR = "teal"             # sampled and validation data points
ACCENT_COLOR = "darkred"        # parity line, OCP feasible box

# ----- timing chart ----------------------------------------------------
BUILD_COLOR = "lightsteelblue"
SOLVE_COLOR = "steelblue"

# ----- lines drawn on several charts --------------------------------------
REF_KW = {"ls": "--", "color": "black", "lw": 2.2,      # mechanistic reference
          "label": "mechanistic", "zorder": 10}
BOUND_KW = {"ls": ":", "color": "grey", "lw": 1}         # variable bounds


def layer_kw(k, lw0=3.2, ms0=8.0):
    """Line width and marker size for the k-th of several lines that may lie
    exactly on top of each other. Each later line is thinner with a smaller
    marker, so lines that coincide stay visible as outlines around each other."""
    return {"lw": max(0.8, lw0 - 0.7 * k), "ms": max(2.5, ms0 - 1.5 * k)}


def ylabel(name):
    """Axis label with units, e.g. 'S  (g/L)'."""
    return f"{name}  ({UNITS[name]})" if name in UNITS else name

---
# 1 &nbsp;&nbsp;The base model

$$
\begin{aligned}
\min_{U(t)}\quad & -P(t_f)\,V(t_f) \\[6pt]
\text{s.t.}\quad
& \frac{\mathrm{d}V}{\mathrm{d}t} = \frac{U}{S_F} \\[4pt]
& \frac{\mathrm{d}X}{\mathrm{d}t} = \mu X - \frac{X}{S_F V}\,U \\[4pt]
& \frac{\mathrm{d}P}{\mathrm{d}t} = \rho X - K_{deg} P - \frac{P}{S_F V}\,U \\[4pt]
& \frac{\mathrm{d}S}{\mathrm{d}t} = -\mu\frac{X}{Y_{X/S}} - \rho\frac{X}{Y_{P/S}}
    - \frac{m_S S}{K_m + S}X + \left(1 - \frac{S}{S_F}\right)\frac{U}{V}
    \qquad \leftarrow \text{the equation the network will learn} \\[6pt]
& \mu = \mu_{max}\frac{S}{K_x X + S},
  \qquad
  \rho = \rho_{max}\frac{S}{K_p + S\left(1 + \frac{S}{K_{in}}\right)} \\[6pt]
& V(0)=7,\quad X(0)=1.5,\quad P(0)=0,\quad S(0)=0 \\[4pt]
& 0 \le U(t) \le 50, \qquad 6.5 \le V(t) \le 10, \qquad 0 \le X(t) \le 40,
  \qquad 0 \le S(t) \le 15, \qquad P(t) \ge 0
\end{aligned}
$$

$V$ is the broth volume (L), $X$ the biomass (g/L), $P$ the penicillin (g/L), $S$ the
substrate (g/L), and $U$ the substrate feed rate (L/h). The objective is the negative of
the penicillin **mass** $P\,V$ at $t_f = 132$ h, so a *more negative* number is a better
batch. $\mu$ and $\rho$ are algebraic variables, defined at every time point by their
own constraints.

### Why the problem is interesting

Two nonlinearities pull against each other:

* **Growth** $\mu = \mu_{max} S/(K_x X + S)$ increases with $S$. More substrate, more
  biomass.
* **Production** $\rho = \rho_{max} S / (K_p + S + S^2/K_{in})$ is *inhibited* at high
  $S$. It peaks at $S^\star = \sqrt{K_p K_{in}} \approx 3.2\times10^{-3}$ g/L and decays
  like $1/S$ after that.

So the optimal feed policy grows biomass early and then holds $S$ very low while the
biomass makes product. The binding constraint is the vessel: $V \le 10$ L with $V(0)=7$
caps the total substrate that can ever be fed at $3\,\text{L} \times S_F = 1500$ g.

### Transcription

`pyomo.dae` turns the ODEs into algebraic constraints by **orthogonal collocation on
finite elements**. With `nfe` elements and `ncp` collocation points per element, the
continuous set is replaced by $\text{nfe} \times \text{ncp} + 1$ time points and each
`DerivativeVar` becomes a linear combination of state values within its element.

The important consequence for this notebook: the ECE formulations build **one copy of
the neural network per time point**, $41$ at `ncp=2` and $61$ at `ncp=3` (the copy at
$t=0$ is built but unused, since $t=0$ has no derivative equation). That linear growth
is outlined in Section 3.

### A note on initialisation

Both $\mu$ and the substrate balance divide by quantities that can be zero:

$$\mu = \mu_{max}\frac{S}{K_x X + S}, \qquad \dots + \left(1-\frac{S}{S_F}\right)\frac{U}{V}$$

At the Pyomo default start (every variable at 0, or pushed to its nearest bound) that is
$0/0$ in $\mu$ and a division by $V=0$ in the last term. IPOPT reports

```
Error in an AMPL evaluation. Run with "halt_on_ampl_error yes" to see details.
Error evaluating Jacobian of equality constraints at user provided starting point.
```

which is exactly what the original sample notebook printed. Two cheap fixes are applied
below:

* every `Var` gets an explicit, physically sensible `initialize=` (`VAR_INIT`);
* `V` gets a lower bound of 6.5 L, so IPOPT's interior-point iterates keep $V > 0$ and
  $U/V$ stays finite. It costs nothing, because $V(0)=7$ and $\dot V = U/S_F \ge 0$, so
  $V$ never decreases. The lower bound of 0 on `X` does the same job for
  $K_x X + S > 0$: IPOPT keeps bounded variables strictly inside their bounds.

The bounds on $\mu$ and $\rho$ are not extra assumptions: $\mu \le \mu_{max}$ since
$K_x X \ge 0$, and $\rho \le \rho_{max}$ since $S/(K_p + S + S^2/K_{in}) \le 1$.

In [ ]:
def build_ocp(nfe=NFE, ncp=NCP_REF, scheme=SCHEME, learned=False):
    """The fed-batch optimal control problem.

    learned=False (mechanistic_model) writes all four state equations.
    learned=True (base_ocp, Section 3.1) leaves out the substrate balance
    ode_S; that is the hole each network embedding fills. Everything else,
    including the kinetics and the other three ODEs, is identical, which is
    why both come from one function.
    """
    m = ConcreteModel()
    m.t = ContinuousSet(bounds=(0, T_FINAL))

    m.U = Var(m.t, initialize=VAR_INIT["U"], bounds=VAR_BOUNDS["U"])
    m.V = Var(m.t, initialize=VAR_INIT["V"], bounds=VAR_BOUNDS["V"])
    m.X = Var(m.t, initialize=VAR_INIT["X"], bounds=VAR_BOUNDS["X"])
    m.P = Var(m.t, initialize=VAR_INIT["P"], bounds=VAR_BOUNDS["P"])
    m.S = Var(m.t, initialize=VAR_INIT["S"], bounds=VAR_BOUNDS["S"])

    # mu and rho are algebraic, not differential (bounds: see the markdown above).
    m.mu = Var(m.t, initialize=0.02, bounds=(0.0, MU_MAX))
    m.rho = Var(m.t, initialize=0.002, bounds=(0.0, RHO_MAX))

    m.dVdt = DerivativeVar(m.V, wrt=m.t, initialize=0.0)
    m.dXdt = DerivativeVar(m.X, wrt=m.t, initialize=0.0)
    m.dPdt = DerivativeVar(m.P, wrt=m.t, initialize=0.0)
    m.dSdt = DerivativeVar(m.S, wrt=m.t, initialize=0.0)

    m.obj = Objective(expr=-m.P[T_FINAL] * m.V[T_FINAL])

    # --- kinetics (algebraic, at every time point) ---
    m.kin_mu = Constraint(
        m.t, rule=lambda M, t:
        M.mu[t] == MU_MAX * M.S[t] / (KX * M.X[t] + M.S[t]))
    m.kin_rho = Constraint(
        m.t, rule=lambda M, t:
        M.rho[t] == RHO_MAX * M.S[t] / (KP + M.S[t] * (1 + M.S[t] / KIN)))

    # --- the known dynamics ---
    # Skipped at t=0: with LAGRANGE-RADAU, dae.collocation writes no derivative
    # equation at the start of the first element, so dVdt[0] ... dSdt[0] are
    # not real degrees of freedom.
    def _fV(M, t):
        return Constraint.Skip if t == 0 else M.dVdt[t] == M.U[t] / SF

    def _fX(M, t):
        return (Constraint.Skip if t == 0
                else M.dXdt[t] == M.mu[t] * M.X[t]
                - (M.X[t] / (SF * M.V[t])) * M.U[t])

    def _fP(M, t):
        return (Constraint.Skip if t == 0
                else M.dPdt[t] == M.rho[t] * M.X[t] - KDEG * M.P[t]
                - (M.P[t] / (SF * M.V[t])) * M.U[t])

    def _fS(M, t):
        return (Constraint.Skip if t == 0
                else M.dSdt[t] == -M.mu[t] * (M.X[t] / YXS)
                - M.rho[t] * (M.X[t] / YPS)
                - (MS * M.S[t] / (KM + M.S[t])) * M.X[t]
                + (1 - M.S[t] / SF) * (M.U[t] / M.V[t]))

    m.ode_V = Constraint(m.t, rule=_fV)
    m.ode_X = Constraint(m.t, rule=_fX)
    m.ode_P = Constraint(m.t, rule=_fP)
    if not learned:
        m.ode_S = Constraint(m.t, rule=_fS)     # <-- the equation the network learns

    TransformationFactory("dae.collocation").apply_to(
        m, nfe=nfe, ncp=ncp, scheme=scheme
    )

    def _init(M):
        yield M.V[0] == V0
        yield M.X[0] == X0
        yield M.P[0] == P0
        yield M.S[0] == S0
        yield ConstraintList.End

    m.init_conditions = ConstraintList(rule=_init)

    # U(0) appears in no constraint of the mechanistic model (the ODEs are not
    # enforced at t=0), so there this fix changes nothing. In the surrogate
    # models it does matter; see the comment in base_ocp (Section 3.1).
    m.U[0].fix(0.0)
    return m


def mechanistic_model(nfe=NFE, ncp=NCP_REF, scheme=SCHEME):
    """The optimal control problem with the TRUE right-hand side (not approximated by an NN).

    This is the reference every surrogate formulation is measured against.
    """
    return build_ocp(nfe, ncp, scheme, learned=False)


PROFILES = STATE_ORDER + ["U", "mu", "rho"]
DERIVS = ["dVdt", "dXdt", "dPdt", "dSdt"]


def trajectories(m):
    """Pull every profile off a solved model, sorted by time.

    mu, rho and the derivative variables are included because warm_start()
    copies them onto the surrogate models in Section 3.
    """
    ts = sorted(m.t)
    out = {"t": np.array(ts)}
    for nm in PROFILES:
        var = getattr(m, nm)
        out[nm] = np.array([value(var[t]) for t in ts])
    for nm in DERIVS:
        var = getattr(m, nm)
        # dSdt[0] and friends are in no constraint and may have no value.
        vals = [value(var[t], exception=False) for t in ts]
        out[nm] = np.array([0.0 if v is None else v for v in vals])
    return out


def active_bounds(traj, tol=1e-6):
    """Names of the OCP bounds the solution touches. t=0 is skipped: the states
    are fixed by the initial conditions there and U(0) is fixed."""
    hits = []
    for nm in STATE_ORDER + ["U"]:
        v = traj[nm][1:]
        lo, hi = VAR_BOUNDS[nm]
        if lo is not None and v.min() < lo + tol:
            hits.append(f"{nm} >= {lo}")
        if hi is not None and v.max() > hi - tol:
            hits.append(f"{nm} <= {hi}")
    return hits


mech = mechanistic_model()
res = SolverFactory("ipopt").solve(mech, tee=False)
REF = trajectories(mech)
REF_OBJ = value(mech.obj)

print(f"status     {res.solver.termination_condition}")
print(f"objective  -P(tf)*V(tf) = {REF_OBJ:.8f}")
print(f"           P(tf) = {REF['P'][-1]:.6f} g/L,  V(tf) = {REF['V'][-1]:.6f} L")
print(f"time pts   {len(REF['t'])}   (nfe={NFE} x ncp={NCP_REF} + 1)")
for nm in STATE_ORDER + ["U"]:
    print(f"{nm} range    [{REF[nm].min():9.4g}, {REF[nm].max():9.4g}]   "
          f"bounds {VAR_BOUNDS[nm]}")
print(f"bounds active: {active_bounds(REF) or 'none'}")
print(f"S over the second half of the batch: median {np.median(REF['S'][REF['t'] > T_FINAL / 2]):.2e} g/L"
      f"   (S* = sqrt(Kp*Kin) = {np.sqrt(KP * KIN):.2e})")

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(14, 6.6))
ax = ax.ravel()

for a, nm in zip(ax[:4], STATE_ORDER):
    a.plot(REF["t"], REF[nm], "-", ms=3, label=nm, **STATE_STYLE[nm])
    a.set_xlabel("time (h)"); a.set_ylabel(ylabel(nm))
    a.set_title(f"{nm}", loc="left")
ax[0].axhline(V_BOUNDS[1], **BOUND_KW)

# S again on a log axis: the linear panel hides where the batch spends most of
# its time, at S ~ 1e-3, five orders of magnitude below the feed peak.
a = ax[4]
a.semilogy(REF["t"][1:], np.maximum(REF["S"][1:], 1e-12), "-",
           ms=3, **STATE_STYLE["S"])
a.axhline(np.sqrt(KP * KIN), **BOUND_KW)
a.text(2, np.sqrt(KP * KIN) * 1.3, r"$S^\star$", color="grey", fontsize=8)
a.set_xlabel("time (h)"); a.set_ylabel(ylabel("S") + ", log axis")
a.set_title("S, log axis", loc="left")

a = ax[5]
a.step(REF["t"], REF["U"], where="post", color=CONTROL_COLOR, lw=1.6)
a.plot(REF["t"], REF["U"], "o", ms=3, color=CONTROL_COLOR)
a.axhline(U_BOUNDS[0], **BOUND_KW)
a.axhline(U_BOUNDS[1], **BOUND_KW)
a.set_xlabel("time (h)"); a.set_ylabel(ylabel("U"))
a.set_title("control U", loc="left")

fig.suptitle(f"Reference solution, known dynamics   $-P(t_f)V(t_f) = {REF_OBJ:.4f}$",
             y=1.0)
fig.tight_layout()
plt.show()

### How much does the collocation order matter?

Before we introduce any surrogate error, we should know how much of the objective
moves purely because of the discretization. Otherwise a difference between two
formulations at different `ncp` is largely uninterpretable. This also helps us decide
when we have "enough" collocation points / finite elements.

Each `ncp` gets its own reference objective, and that is what the surrogate results get
compared against later to make the comparisons fair. It is also the warm start each
surrogate model at that `ncp` is given (below), so the two grids never get mixed.

In [ ]:
ref_rows = []
REF_BY_NCP = {}

for ncp in NCP_LIST:
    mm = mechanistic_model(ncp=ncp)
    rr = SolverFactory("ipopt").solve(mm, tee=False)
    REF_BY_NCP[ncp] = {"traj": trajectories(mm), "obj": value(mm.obj)}
    ref_rows.append({
        "ncp": ncp,
        "time points": len(mm.t),
        "objective -P*V": value(mm.obj),
        "P(tf)": value(mm.P[T_FINAL]),
        "V(tf)": value(mm.V[T_FINAL]),
        "status": str(rr.solver.termination_condition),
    })

ref_df = pd.DataFrame(ref_rows)
# Difference from the next-coarser ncp (NCP_LIST is sorted ascending), i.e. how
# far the objective moves from the discretization alone. NaN for the first row.
ref_df["Δ obj vs previous ncp"] = ref_df["objective -P*V"].diff()
print(ref_df.to_string(index=False))

### The starting point picks the optimum

This problem has more than one local optimum. Solve the same mechanistic model from a
start of all zeros (the Pyomo default) instead of `VAR_INIT`, and IPOPT still reports
`optimal`, but at a different, worse objective.

That matters for every comparison in Section 4: two formulations that start from
different points can land in different local optima, and the difference would look
like a formulation effect. So by default (`WARM_START = True`) every surrogate model is
started from the mechanistic solution at the same `ncp`. Every formulation then
starts from the same point, and the iteration counts compare embeddings rather than
starting points.

This warm start is a convenience of the tutorial, not something available in practice:
it uses the true $\dot S$, which is the thing we pretend not to know. Set
`WARM_START = False` to start every surrogate model from `VAR_INIT` instead.

In [ ]:
def zero_start(m):
    """Set every free variable to 0, the Pyomo default (pushed into its bounds
    by IPOPT)."""
    for v in m.component_data_objects(Var, active=True):
        if not v.fixed:
            v.value = 0.0
    return m


start_rows = []
for label, prep in (("VAR_INIT", lambda m: m), ("all zeros", zero_start)):
    mm = prep(mechanistic_model(ncp=NCP_REF))
    rr = SolverFactory("ipopt").solve(mm, tee=False)
    start_rows.append({"start": label,
                       "status": str(rr.solver.termination_condition),
                       "objective -P*V": value(mm.obj),
                       "P(tf)": value(mm.P[T_FINAL]),
                       "V(tf)": value(mm.V[T_FINAL])})

print(f"mechanistic model, ncp={NCP_REF}, two starting points\n")
print(pd.DataFrame(start_rows).to_string(index=False))

---
# 2 &nbsp;&nbsp;The surrogate

Now we throw away **one** equation and learn it. In CS1 and CS2 the network learned the
derivative of every state; here it learns only the substrate balance:

$$
\mathcal{N}: \; (X,\; V,\; S,\; U) \;\longmapsto\; \dot S
$$

$\dot V$, $\dot X$, $\dot P$, $\mu$ and $\rho$ stay mechanistic in every model. Note what
is *not* an input: $P$ does not appear in the true substrate balance, so it is not
offered to the network. Feeding it anyway would add a variable and a linking constraint
at every collocation point, and would give the optimizer a false path through which $P$
could change $\dot S$.

### Sampling

The training points come from a **Latin hypercube** over the input space.

**NOTE: The sampling space is larger than the OCP space** on every face where the
target exists: $X \in [0, 48]$ vs $[0, 40]$, $V \in [6, 10.5]$ vs $[6.5, 10]$,
$S \in [0, 20]$ vs $[0, 15]$, and $U \in [-3, 55]$ vs $[0, 50]$. If the optimum presses
against a bound, that is where a network trained only up to the bound starts to
struggle. $U$ is widened below zero even though negative feed has no physical meaning:
the formula for $\dot S$ is still defined there, and the network has to be accurate
right at $U = 0$. The floors of $X$ and $S$ are not widened, because the target does
not exist below them: $\mu = \mu_{max}S/(K_xX+S)$ is singular on $K_x X + S = 0$, and
the maintenance term is singular at $S = -K_m$.

Targets are the analytic derivatives, noise-free. That is a deliberate simplification:
it means any objective mismatch between formulations in Section 4 is a code/formulation
bug, not NN model error. With noisy targets it becomes difficult to tell the two apart.

In [ ]:
def true_derivatives(X):
    """Analytic substrate balance dS/dt, the one equation the network learns.

    Reads the columns of X by name (INPUT_ORDER) and returns the outputs in
    OUTPUT_ORDER, so the labels stay correct whatever order the lists use.
    """
    v = dict(zip(INPUT_ORDER, X.T))
    Xb, V, S, U = v["X"], v["V"], v["S"], v["U"]

    # mu is 0/0 at X = S = 0 exactly. The design never lands there, but a
    # guard costs nothing.
    den = KX * Xb + S
    mu = np.where(den > 0, MU_MAX * S / np.where(den > 0, den, 1.0), 0.0)
    rho = RHO_MAX * S / (KP + S * (1 + S / KIN))

    rhs = {
        "dSdt": (-mu * Xb / YXS - rho * Xb / YPS
                 - (MS * S / (KM + S)) * Xb + (1 - S / SF) * (U / V)),
    }
    return np.column_stack([rhs[nm] for nm in OUTPUT_ORDER])


def sample_lhs(n=N_SAMPLES, seed=SEED):
    """Latin hypercube over SAMPLE_BOUNDS, returned in INPUT_ORDER.

    criterion="center" rather than "maximin": at 40 000 points the maximin
    search would dominate this section's runtime and buys nothing for a
    noise-free target.
    """
    np.random.seed(seed)
    lo = np.array([SAMPLE_BOUNDS[nm][0] for nm in INPUT_ORDER])
    hi = np.array([SAMPLE_BOUNDS[nm][1] for nm in INPUT_ORDER])
    unit = lhs(N_IN, samples=n, criterion="center")
    return lo + unit * (hi - lo)


X_all = sample_lhs()
Y_all = true_derivatives(X_all)

n_train = int((1 - VAL_FRACTION) * len(X_all))
X_train, X_val = X_all[:n_train], X_all[n_train:]
Y_train, Y_val = Y_all[:n_train], Y_all[n_train:]

print(f"samples   {X_all.shape[0]}   train {len(X_train)} / val {len(X_val)}")
print(f"inputs    {INPUT_ORDER}   ranges {[SAMPLE_BOUNDS[n] for n in INPUT_ORDER]}")
print(f"outputs   {OUTPUT_ORDER}")
print()
print(pd.DataFrame(
    np.column_stack([X_all, Y_all]),
    columns=INPUT_ORDER + OUTPUT_ORDER,
).describe().loc[["min", "mean", "max"]].to_string())

In [ ]:
# Every pair of inputs, three panels per row. Every 20th sample is drawn, so
# the 40 000 points do not turn into a solid block.
pairs = list(combinations(range(N_IN), 2))
ncols = min(3, len(pairs))
nrows = int(np.ceil(len(pairs) / ncols))
fig, ax = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.3 * nrows), squeeze=False)
ax = ax.ravel()
for a, (i, j) in zip(ax, pairs):
    ni, nj = INPUT_ORDER[i], INPUT_ORDER[j]
    a.scatter(X_all[::20, i], X_all[::20, j], s=5, alpha=0.45, color=DATA_COLOR)
    a.set_xlabel(ylabel(ni)); a.set_ylabel(ylabel(nj))
    # Show the OCP feasible space inside the (wider) sampling space.
    (xlo, xhi), (ylo, yhi) = VAR_BOUNDS[ni], VAR_BOUNDS[nj]
    a.add_patch(plt.Rectangle((xlo, ylo), xhi - xlo, yhi - ylo,
                              fill=False, ec=ACCENT_COLOR, lw=1.6, ls="--"))
for a in ax[len(pairs):]:
    a.set_visible(False)
fig.suptitle(f"LHS design, {N_SAMPLES} points, every 20th shown "
             f"(dashed = OCP feasible space)", y=1.0)
fig.tight_layout()
plt.show()

### Training, and saving

We train two networks with identical architecture and data, differing only in the
hidden activation. That is the point: the activation alone decides *which
formulations are even available* in Section 3.

The networks here have two hidden layers ($4 \to 40 \to 40 \to 1$) where CS1 and CS2 use
one, because $\dot S$ is a harder function to fit. Training uses early stopping on the
validation loss and halves the learning rate when it stalls, so `EPOCHS` is an upper
limit.

A practical note on saving: A `.keras` file records the exact layer config of the
Keras version that wrote it, and newer versions add keys (`quantization_config`, for
one) that older versions do not load. So, alongside the `.keras` file we save a plain
`.npz` of the weight arrays plus the architecture, and rebuild from that for redundancy.

In [ ]:
def build_net(activation, n_hidden=N_HIDDEN, n_layers=N_LAYERS):
    """N_LAYERS hidden layers, linear output.

    An explicit Input layer is required: passing input_shape= to Dense is
    removed in Keras 3. Compiled here rather than in train_net so that a
    network rebuilt by load_net is also ready for evaluate().
    """
    model = k.Sequential(
        [k.layers.Input(shape=(N_IN,))]
        + [k.layers.Dense(n_hidden, activation=activation) for _ in range(n_layers)]
        + [k.layers.Dense(N_OUT)]
    )
    model.compile(optimizer=k.optimizers.Adam(1e-3), loss="mean_squared_error")
    return model


def training_settings():
    """Everything (besides activation and N_HIDDEN, which are in the file
    name) that decides what a trained network is. Saved with the weights; a
    saved network is reused only if every entry still matches."""
    return {
        "input_order": list(INPUT_ORDER),
        "output_order": list(OUTPUT_ORDER),
        "n_layers": N_LAYERS,
        "sample_bounds": {nm: list(SAMPLE_BOUNDS[nm]) for nm in INPUT_ORDER},
        "n_samples": N_SAMPLES,
        "val_fraction": VAL_FRACTION,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "patience": [PATIENCE, LR_PATIENCE],
        "seed": SEED,
    }


def save_net(model, activation, n_hidden=N_HIDDEN):
    """Save both the native format and a version-proof weight archive."""
    model.save(KERAS_FMT.format(act=activation, n=n_hidden))
    arrays = {f"w{i}": w for i, w in enumerate(model.get_weights())}
    np.savez(
        WEIGHTS_FMT.format(act=activation, n=n_hidden),
        activation=activation, n_hidden=n_hidden, n_layers=N_LAYERS,
        n_arrays=len(arrays),
        settings=json.dumps(training_settings(), sort_keys=True),
        **arrays,
    )


def load_net(activation, n_hidden=N_HIDDEN):
    """Rebuild from the .npz archive. Independent of the Keras version."""
    with np.load(WEIGHTS_FMT.format(act=activation, n=n_hidden)) as d:
        model = build_net(d["activation"].item(), int(d["n_hidden"]),
                          int(d["n_layers"]))
        model.set_weights([d[f"w{i}"] for i in range(int(d["n_arrays"]))])
    return model


def cache_mismatch(activation, n_hidden=N_HIDDEN):
    """Why the saved network cannot be reused, or None if it can.

    If the notebook has been run before, the saved network is loaded to
    avoid retraining. It is only reused if it was trained with the current
    training_settings() (variable order, sampling box, sample count,
    architecture, epochs, ...). If an edit has changed any of them, this
    returns the names of what changed, and the network is retrained and
    resaved.
    """
    path = WEIGHTS_FMT.format(act=activation, n=n_hidden)
    if not os.path.exists(path):
        return "no saved network"
    with np.load(path) as d:
        if "settings" not in d.files:
            return "saved before training settings were recorded"
        saved = json.loads(d["settings"].item())
    now = json.loads(json.dumps(training_settings(), sort_keys=True))
    changed = [key for key in now if saved.get(key) != now[key]]
    return f"changed: {', '.join(changed)}" if changed else None


def train_net(activation, epochs=EPOCHS, batch_size=BATCH_SIZE):
    tf.keras.utils.set_random_seed(SEED)
    model = build_net(activation)
    callbacks = [
        k.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                  restore_best_weights=True),
        # The fixed 1e-3 step stops improving the fit long before EPOCHS;
        # halving it on a plateau turns the extra epochs into accuracy.
        k.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                      patience=LR_PATIENCE, min_lr=1e-6),
    ]
    t0 = time.perf_counter()
    hist = model.fit(X_train, Y_train, epochs=epochs, batch_size=batch_size,
                     validation_data=(X_val, Y_val), callbacks=callbacks, verbose=0)
    return model, hist, time.perf_counter() - t0


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


NETS, HISTORY, train_rows = {}, {}, []

for act in ACTIVATIONS:
    npz = WEIGHTS_FMT.format(act=act, n=N_HIDDEN)
    reason = "FORCE_RETRAIN" if FORCE_RETRAIN else cache_mismatch(act)
    if reason is None:
        NETS[act] = load_net(act)
        wall, n_epochs = np.nan, np.nan
        print(f"{act:5s}  loaded {npz}")
    else:
        print(f"{act:5s}  retraining ({reason})")
        print(f"{act:5s}  training up to {EPOCHS} epochs ...", flush=True)
        NETS[act], HISTORY[act], wall = train_net(act)
        n_epochs = len(HISTORY[act].history["loss"])
        save_net(NETS[act], act)
        print(f"{act:5s}  done in {wall:.1f}s after {n_epochs} epochs  ->  {npz}")

    train_rows.append({
        "activation": act,
        "train MSE": NETS[act].evaluate(X_train, Y_train, verbose=0),
        "val MSE": NETS[act].evaluate(X_val, Y_val, verbose=0),
        "val RMSE (g/L/h)": rmse(NETS[act].predict(X_val, verbose=0), Y_val),
        "parameters": NETS[act].count_params(),
        "epochs": n_epochs,
        "train time (s)": wall,
    })

print()
print(pd.DataFrame(train_rows).to_string(index=False))
print(f"\nfor scale: std(dS/dt) over the design = {Y_all.std():.3f} g/L/h")

In [ ]:
# Parity plots for model validation
fig, ax = plt.subplots(1, len(ACTIVATIONS), figsize=(4.6 * len(ACTIVATIONS), 3.9))
ax = np.atleast_1d(ax)
for a, act in zip(ax, ACTIVATIONS):
    P = NETS[act].predict(X_val, verbose=0)
    a.scatter(Y_val[:, 0], P[:, 0], s=4, alpha=0.25, color=DATA_COLOR)
    lim = [min(Y_val.min(), P.min()), max(Y_val.max(), P.max())]
    a.plot(lim, lim, "--", color=ACCENT_COLOR, lw=1)
    a.set_title(f"{act}  {OUTPUT_ORDER[0]}   RMSE={rmse(Y_val, P):.2e}", fontsize=9)
    a.set_xlabel("true " + ylabel("dSdt")); a.set_ylabel("predicted")
fig.suptitle("Validation parity", y=1.0)
fig.tight_layout()
plt.show()

Both networks fit the validation set well: an RMSE of a few hundredths of a g/L/h
against a spread of about 4 g/L/h in $\dot S$, with `tanh` ahead of `relu` on the same
data and budget. That number is an average over the sampling box, though, and the
optimizer does not visit the box evenly: Section 1 showed $S$ sitting near
$2\times10^{-3}$ g/L for the whole second half of the batch, a thin corner of a box that
runs to 20 g/L. Keep that in mind for Section 4, where both networks' optima land well
short of the mechanistic one.

---
# 3 &nbsp;&nbsp;Four ways to embed the same network

Every formulation below solves *the same optimization problem*. They differ only in
what the solver is shown.

$$
\dot S(t) = \mathcal{N}\big(X(t),\, V(t),\, S(t),\, U(t)\big)
\qquad \forall t \in \mathcal{T}_{\text{col}}
$$

with $\dot V$, $\dot X$, $\dot P$, $\mu$ and $\rho$ still written out mechanistically.

### The design space

For a network with $L$ hidden layers of $H$ nodes evaluated at $N$ collocation points:

| | added variables | added constraints | derivatives from | needs |
|---|---|---|---|---|
| `ECE-FS`  | $\mathcal{O}(N\,L\,H)$ | $\mathcal{O}(N\,L\,H)$ | AMPL AD on the algebra | any NLP solver |
| `ECE-RS`  | $\mathcal{O}(N)$ | $\mathcal{O}(N)$ | AMPL AD on nested expressions | any NLP solver |
| `ECE-CMP` | $\mathcal{O}(N\,L\,H)$ | $\mathcal{O}(N\,L\,H)$ | AMPL AD on the algebra | any NLP solver |
| `EFE`     | $\mathcal{O}(N)$ | $\mathcal{O}(N)$ | TensorFlow, by callback | cyipopt + PyNumero |

The trade is between problem size and what computes the derivatives.
`ECE` hands the solver an enormous but perfectly sparse, mostly-linear problem.
`EFE` hands it a smaller problem and a derivative evaluator to call.

**`ECE-RS` and `EFE` are both $\mathcal{O}(N)$, but do not have the same number of
variables.** The cause is OMLT: every OMLT formulation, full space, reduced space, or
complementarity, unconditionally builds a `scaled_inputs` / `scaled_outputs` pair and
two linking constraints per block, **even with no scaler configured**
(`omlt/formulation.py`, `_setup_scaled_inputs_outputs`; with no scaling object the
"scaling" is just the identity `scaled_inputs == inputs`, but the variables and
constraints are still real Pyomo objects). PyNumero's grey box has no such layer. So per
collocation point, with $n_{in}=4,\ n_{out}=1$:

$$
\underbrace{n_{in} + n_{out} + \overbrace{n_{in} + n_{out}}^{\text{scaled}}}_{\text{ECE-RS: 10 vars/point}}
\qquad\text{vs.}\qquad
\underbrace{n_{in} + n_{out}}_{\text{EFE: 5 vars/point}}
$$

The table above shows how each formulation *scales*; it does not say the
$\mathcal{O}(N)$ rows are numerically close, see Section 4.1's measured table.

With a single output the network is a small part of the NLP for the $\mathcal{O}(N)$
formulations: the mechanistic part (four states, $U$, $\mu$, $\rho$, three ODEs and the
kinetics) is the same in every model and is a large share of the variables.

### The support matrix

Not every activation works with every formulation. This table is not a summary of how
OMLT and PyNumero behave. It is what happens when you build each
`(formulation, activation)` pair with no `activation_constraints` override.

| | `tanh` | `relu` |
|---|---|---|
| `ECE-FS`  | smooth NLP | big-M, introduces binaries, becomes an MINLP, not tested in this work |
| `ECE-RS`  | smooth NLP | **unsupported** (no closed-form smooth composition) |
| `ECE-CMP` | **unsupported** (ReLU-only formulation) | smooth NLP via complementarity |
| `EFE`     | smooth NLP | works, but the surrogate is not continuously differentiable, not tested in this work |

## 3.1 &nbsp;The shared skeleton

Everything except the substrate equation is identical across all four methods.

`base_ocp` is `build_ocp` from Section 1 with `learned=True`: the same model without the
`ode_S` constraint. That is the hole each embedding fills. `kin_mu`, `kin_rho`, `ode_V`,
`ode_X` and `ode_P` all stay, so $\mu$ and $\rho$ are still computed from the true
kinetics, and they use the $S$ that the surrogate produces.

In [ ]:
def base_ocp(nfe=NFE, ncp=NCP_REF, scheme=SCHEME):
    """The OCP with the substrate balance left out.

    Identical to mechanistic_model() minus the ode_S constraint; every
    formulation below starts from this and adds its own way of supplying dSdt.
    """
    m = build_ocp(nfe, ncp, scheme, learned=True)

    # Every formulation builds a network copy at t=0 and links U(0) to its
    # inputs, but that copy's output feeds nothing (t=0 has no derivative
    # equation). That pulls U(0) into the problem with no effect on the
    # objective: a degenerate, non-unique variable that IPOPT would park
    # mid-bounds. build_ocp already fixes it, which removes the degeneracy.
    # U(0) is therefore not an optimal control value; the 0 in the control
    # plots is this fixed value.
    assert m.U[0].fixed
    return m


def warm_start(m, traj):
    """Copy a solved trajectory onto a fresh model's DAE variables.

    The reference and the surrogate model share nfe, ncp and scheme, so their
    time grids are identical and this is a copy, not an interpolation. That is
    checked rather than assumed: a grid mismatch would give a bad start that
    looks like a slow formulation.

    Only the DAE-level variables are set here. The network's own variables
    are set by build_ece / build_efe below.
    """
    ts = sorted(m.t)
    if len(ts) != len(traj["t"]) or not np.allclose(ts, traj["t"]):
        raise ValueError("warm_start: time grids differ, refusing to interpolate")
    for i, t in enumerate(ts):
        for nm in PROFILES + DERIVS:
            v = getattr(m, nm)[t]
            if not v.fixed:
                v.value = float(traj[nm][i])
    return m


def constraint_names(m):
    return {c.local_name for c in m.component_objects(Constraint, active=True)}


_mech, _probe = mechanistic_model(ncp=NCP_REF), base_ocp(ncp=NCP_REF)
print(f"skeleton at ncp={NCP_REF}:  {len(_probe.t)} time points, "
      f"{sum(1 for _ in _probe.component_data_objects(Var, active=True)):d} variables")
print(f"in the mechanistic model but not in the skeleton: "
      f"{sorted(constraint_names(_mech) - constraint_names(_probe))}")
print("dSdt is currently unconstrained, each formulation fills it in.")

## 3.2 &nbsp;ECE: explicit constraint embedding with OMLT

OMLT reads the Keras model and writes the network into Pyomo as algebra. We build
**one `OmltBlock` per collocation point** and wire it to the DAE variables with linking
constraints, four in and one out per point.

**Starting values inside the block.** A warm start only sets the DAE variables. The
full-space and complementarity blocks also carry a variable for every neuron, and left
alone those start at 0, far from where everything else starts. `seed_omlt_block` writes
a forward pass of the network at the starting point onto them, so every formulation
starts from the same, consistent point. On this problem that roughly halves the
iteration count of the full-space and complementarity solves.

### Full space (`FullSpaceNNFormulation`)

Every neuron gets variables and constraints:

$$
\hat z^{(l)} = W^{(l)} z^{(l-1)} + b^{(l)}, \qquad z^{(l)} = \sigma\big(\hat z^{(l)}\big)
$$

with $\hat z$ (pre-activation) and $z$ (post-activation) both explicit Pyomo variables.
Enormous, but every constraint is either linear or a single scalar nonlinearity. This
results in a sparse Jacobian, and the solver's linear algebra is cheap per iteration.

### Reduced space (`ReducedSpaceNNFormulation`)

The same network as one substituted expression per output:

$$
z^{(L)} = W^{(L)}\,\sigma\big(W^{(L-1)}\sigma(\cdots) + b^{(L-1)}\big) + b^{(L)}
$$

No intermediate variables at all. A far smaller problem, but each constraint is a deep
nested expression whose gradient is dense in the inputs. With two hidden layers the
nesting is one level deeper than in CS1.

### ReLU complementarity (`ReluComplementarityFormulation`)

$z = \max(0, \hat z)$ has no derivative at the kink, so the default full-space handling
of ReLU is **big-M with a binary per neuron**. This turns the NLP into an MINLP. The
complementarity formulation instead writes

$$
z \ge 0, \qquad z - \hat z \ge 0, \qquad z\,(z - \hat z) = 0
$$

which is smooth and continuous, so an NLP solver can attack it directly. The price is
that complementarity constraints violate the usual constraint qualifications at the
solution, and introduce nonconvexity, which is why these solves take more iterations
than their size suggests.

### Input bounds

`load_keras_sequential` takes `unscaled_input_bounds`, and OMLT propagates them through
the network to bound the hidden-layer variables (the complementarity bounds, and the
big-M coefficients). Here $U$ spans $[0,\,50]$ and $X$ $[0,\,40]$ while $V$ spans an
interval of width $3.5$, so the propagated bounds are loose, which is one more reason the
complementarity solves are harder than their size suggests.

In [ ]:
# OMLT identifies inputs by position only, so key the bounds by the position
# each name has in INPUT_ORDER.
OMLT_INPUT_BOUNDS = {i: VAR_BOUNDS[nm] for i, nm in enumerate(INPUT_ORDER)}

ECE_FORMULATIONS = {
    "ECE-FS": FullSpaceNNFormulation,
    "ECE-RS": ReducedSpaceNNFormulation,
    "ECE-CMP": ReluComplementarityFormulation,
}


def make_network(name):
    """Keras model -> OMLT network definition."""
    return load_keras_sequential(NETS[name], unscaled_input_bounds=OMLT_INPUT_BOUNDS)


ACTIVATION_NP = {"tanh": np.tanh, "relu": lambda v: np.maximum(v, 0.0),
                 "linear": lambda v: v}


def seed_omlt_block(blk, net, z_in):
    """Write a forward pass of `net` at input z_in onto the block.

    Full-space and complementarity blocks hold a Var for every neuron (z, and
    zhat before the activation); reduced-space blocks hold expressions there,
    which need no value, so only Vars are set. Dense layers only, so every
    index is a plain integer.
    """
    x = np.asarray(z_in, dtype=np.float64)
    vals = {}
    for layer in net.layers:
        if isinstance(layer, InputLayer):
            pre = out = x
        else:
            prev = vals[id(next(iter(net.predecessors(layer))))]
            pre = prev @ np.asarray(layer.weights) + np.asarray(layer.biases)
            out = ACTIVATION_NP[layer.activation](pre)
        vals[id(layer)] = out
        lb = blk.layer[id(layer)]
        for comp, arr in (("z", out), ("zhat", pre)):
            var = getattr(lb, comp, None)
            if var is not None and var.ctype is Var:
                for i, v in enumerate(np.ravel(arr)):
                    var[i].value = float(v)
    for j in range(N_IN):
        blk.inputs[j].value = blk.scaled_inputs[j].value = float(x[j])
    for o in range(N_OUT):
        blk.outputs[o].value = blk.scaled_outputs[o].value = float(np.ravel(out)[o])


def build_ece(name, formulation_name, nfe=NFE, ncp=NCP_REF, init=None):
    """One OMLT block per collocation point, linked to the DAE variables.

    `name` is a key of NETS ("tanh", "relu", ...). `init` is a trajectory to
    warm start from, or None for the VAR_INIT start.

    Raises ValueError from OMLT if the activation is unsupported by the
    requested formulation - that is informative, so it is not caught here.
    """
    m = base_ocp(nfe, ncp)
    net = make_network(name)
    formulation = ECE_FORMULATIONS[formulation_name](net)

    m.nn = Block(m.t)
    for t in m.t:
        m.nn[t].m = OmltBlock()
        m.nn[t].m.build_formulation(formulation)

    # --- linking: DAE variables <-> network inputs ---
    # OMLT numbers its inputs 0..N_IN-1; INPUT_ORDER says which Pyomo
    # variable sits at each position.
    m.link_in = Constraint(
        m.t, range(N_IN),
        rule=lambda M, t, i: getattr(M, INPUT_ORDER[i])[t]
        == M.nn[t].m.inputs[i])

    # --- linking: network output -> derivative ---
    # Skipped at t=0: dae.collocation writes no derivative equation there,
    # so dSdt[0] is not a real degree of freedom.
    m.link_out = Constraint(
        m.t, range(N_OUT),
        rule=lambda M, t, o: Constraint.Skip if t == 0
        else getattr(M, OUTPUT_ORDER[o])[t] == M.nn[t].m.outputs[o])

    # --- starting point ---
    # The DAE variables start at the warm start (or VAR_INIT); every block
    # gets a forward pass at the same point, so the whole model starts
    # consistent (see the markdown above).
    traj = init if init is not None else {nm: np.full(len(m.t), VAR_INIT[nm])
                                          for nm in INPUT_ORDER}
    if init is not None:
        warm_start(m, init)
    X0_in = np.column_stack([traj[nm] for nm in INPUT_ORDER])
    for i, t in enumerate(sorted(m.t)):
        seed_omlt_block(m.nn[t].m, net, X0_in[i])
    return m


print("OMLT input bounds (position -> name, bounds):")
for i, nm in enumerate(INPUT_ORDER):
    print(f"  {i}  {nm}  {OMLT_INPUT_BOUNDS[i]}")

### Check the wiring before optimising anything

Everything downstream assumes that the output of one block equals `model.predict(x)`
for the same `x`. That is three things agreeing at once: `INPUT_ORDER`, the OMLT input
positions, and the Keras column order. If any one of them is off, nothing raises. The
optimizer just solves a different problem.

So the check below fixes the inputs of one block, solves the resulting square system,
and compares against Keras. It queries $S$ across several decades, including $S = 0$,
the initial condition.

In [ ]:
def block_prediction(name, formulation_name, Xq):
    """Evaluate one OMLT block at each row of Xq (inputs in INPUT_ORDER)."""
    net = make_network(name)
    mm = ConcreteModel()
    mm.m = OmltBlock()
    mm.m.build_formulation(ECE_FORMULATIONS[formulation_name](net))
    mm.obj = Objective(expr=0.0)

    solver = SolverFactory("ipopt")
    solver.options["tol"] = 1e-10
    out = np.zeros((len(Xq), N_OUT))
    for r, x in enumerate(Xq):
        for j in range(N_IN):
            mm.m.inputs[j].fix(float(x[j]))
        seed_omlt_block(mm.m, net, x)
        solver.solve(mm)
        out[r] = [value(mm.m.outputs[o]) for o in range(N_OUT)]
    return out


# Query points inside the OCP box, spanning the decades of S.
rng = np.random.default_rng(SEED)
S_check = np.array([0.0, 1e-4, 3e-3, 0.5, 14.0])
_q = {"X": rng.uniform(1.0, X_BOUNDS[1], len(S_check)),
      "V": rng.uniform(*V_BOUNDS, len(S_check)),
      "S": S_check,
      "U": rng.uniform(*U_BOUNDS, len(S_check))}
X_check = np.column_stack([_q[nm] for nm in INPUT_ORDER])

# What limits agreement is NOT the embedding:
#   smooth NLP - Keras evaluates in float32, Pyomo/IPOPT in float64. On
#                outputs of order 1-10 that is a few times 1e-7.
#   ECE-CMP    - z*(z - zhat) = 0 is met to the barrier tolerance, not
#                exactly, so the reconstructed ReLU carries a larger residual.
WIRE_TOL = {"ECE-FS": 1e-5, "ECE-RS": 1e-5, "ECE-CMP": 1e-3}

print("OmltBlock vs keras predict\n")
for name, fname in (("tanh", "ECE-FS"), ("tanh", "ECE-RS"), ("relu", "ECE-CMP")):
    err = np.abs(NETS[name].predict(X_check, verbose=0)
                 - block_prediction(name, fname, X_check)).max()
    flag = "OK" if err < WIRE_TOL[fname] else "MISMATCH - check INPUT_ORDER"
    print(f"  {fname:8s} {name:5s}  max abs difference {err:.3e}   "
          f"(tol {WIRE_TOL[fname]:.0e})   {flag}")
print(f"\n  queried at S = {S_check.tolist()}")

Let's look at what each formulation actually costs, on a *single* network block,
before any of it gets multiplied by the number of collocation points.

In [ ]:
def block_size(name, formulation_name):
    """Variables/constraints/binaries contributed by ONE OMLT block."""
    blk = ConcreteModel()
    blk.m = OmltBlock()
    blk.m.build_formulation(ECE_FORMULATIONS[formulation_name](make_network(name)))
    nv = sum(1 for _ in blk.component_data_objects(Var, active=True,
                                                   descend_into=True))
    nc = sum(1 for _ in blk.component_data_objects(Constraint, active=True,
                                                   descend_into=True))
    nb = sum(1 for v in blk.component_data_objects(Var, active=True,
                                                   descend_into=True)
             if v.domain is Binary)
    return nv, nc, nb


rows = []
for act in ACTIVATIONS:
    for fname in ECE_FORMULATIONS:
        try:
            nv, nc, nb = block_size(act, fname)
            rows.append({"activation": act, "formulation": fname,
                         "vars/block": nv, "cons/block": nc,
                         "binaries/block": nb, "status": "ok"})
        except Exception as exc:
            rows.append({"activation": act, "formulation": fname,
                         "vars/block": np.nan, "cons/block": np.nan,
                         "binaries/block": np.nan,
                         "status": f"{type(exc).__name__}: {exc}"})

print(f"one OMLT block, {ARCH}\n")
print(pd.DataFrame(rows).to_string(index=False))

`ECE-FS` with a **ReLU** network introduces one binary per hidden node, per collocation
point. With two hidden layers of 40 at `ncp=3` that is $80 \times 61 = 4880$ binaries,
and the problem is no longer an NLP. It becomes an integer problem.

The tricky part is what happens next: The `.nl` writer emits the integrality markers,
IPOPT ignores them, and you get back a converged "optimal" solution to the continuous
relaxation. This is the lower bound to the true MINLP objective, and it may not be
flagged depending on settings. You would need a solver capable of handling
mixed-integer problems.

For this reason, ECE-FS with ReLU activation is omitted from this work, as in CS1. To
extend the work, an MINLP solver will need to be used.

## 3.3 &nbsp;EFE: external function embedding with PyNumero

Now the opposite. The network never enters the Pyomo expression tree at all. We
register **one grey-box block** covering every collocation point at once, and IPOPT
calls back into TensorFlow whenever it needs residuals or derivatives.

The block exposes only inputs and outputs, flattened point-major in `INPUT_ORDER` /
`OUTPUT_ORDER` (shown here for the default lists):

```
inputs  = [X_0, V_0, S_0, U_0,   X_1, V_1, S_1, U_1,   ...]     4N values
outputs = [dSdt_0,               dSdt_1,               ...]      N values
```

### How this scales

The block contributes $4N$ (where $N$ is the number of time points) input variables,
$N$ output variables, and $N$ output constraints, and **that is the entire cost,
whatever the network is**. Width, depth and activation change nothing: a 40-40 network
and a 500-500-500 network hand IPOPT exactly the same NLP. Everything the network does
happens inside a callback the solver never sees. That is the structural difference
from ECE, where every hidden node becomes variables and constraints in the model
itself, and it is what Section 4.1 measures.

Using one block rather than $N$ separate 4-in/1-out blocks changes nothing about the
NLP either: same variables, same constraints, same block-diagonal Jacobian sparsity. It
changes only what the callback costs. PyNumero calls back once per block, so $N$ blocks
means $N$ `tf.function` dispatches at batch size 1 per solver iteration, while one block
is a single dispatch at batch size $N$: one `(N, 4)` matrix in, one `(N, 1)` matrix out,
one batched Jacobian. At this network size the arithmetic is negligible and per-call
overhead dominates, so the batched form pays that overhead once per iteration instead
of $N$ times.

### Float64

IPOPT is a double-precision code, and a float32 network's derivatives carry roughly
$10^{-6}$ error - two orders of magnitude coarser than the $10^{-8}$ tolerance the
solver is asked to meet. `to_float64` below rebuilds the network layer by layer; Keras 3
makes the `set_floatx` shortcut unreliable, and its docstring says why.

In [ ]:
def to_float64(model_f32):
    """Rebuild a Dense-only Keras model in float64 and copy the weights.

    Explicit reconstruction, not from_config under set_floatx: Keras 3 embeds
    the dtype policy in each layer config, so the floatx trick returns a
    float32 model and you get "MatMul float32 vs float64" errors later.
    """
    layers64 = []
    for lyr in model_f32.layers:
        if lyr.__class__.__name__ == "InputLayer":
            continue
        if not isinstance(lyr, k.layers.Dense):
            raise TypeError(f"only Dense layers are handled, got {type(lyr)}")
        layers64.append(k.layers.Dense(
            lyr.units, activation=lyr.get_config()["activation"],
            dtype="float64"))

    model64 = k.Sequential(layers64)
    model64.build((None, N_IN))
    model64.set_weights([w.astype(np.float64)
                         for w in model_f32.get_weights()])
    return model64


class TFEvaluator:
    """Batched value / Jacobian / Hessian for a Keras network.

    Every call handles all N collocation points at once. The tf.function
    wrappers are traced on first use; in Section 4 that cost lands in the
    first-run time (t_first_s), not in the repeated-run medians.
    """

    def __init__(self, model):
        self.model = model

        @tf.function(reduce_retracing=True)
        def _fwd(x):
            return self.model(x, training=False)

        @tf.function(reduce_retracing=True)
        def _jac(x):
            with tf.GradientTape() as tape:
                tape.watch(x)
                y = self.model(x, training=False)
            return tape.batch_jacobian(y, x)               # (N, n_out, n_in)

        @tf.function(reduce_retracing=True)
        def _hess(x, lam):
            # Second tape differentiates the lambda-weighted gradient, which
            # is exactly sum_i lam_i * d2f_i/dx2 without forming each output's
            # Hessian separately.
            with tf.GradientTape() as t2:
                t2.watch(x)
                with tf.GradientTape() as t1:
                    t1.watch(x)
                    y = self.model(x, training=False)
                g = t1.gradient(y, x, output_gradients=lam)    # (N, n_in)
            return t2.batch_jacobian(g, x)                     # (N,n_in,n_in)

        self._fwd, self._jac, self._hess = _fwd, _jac, _hess

    @staticmethod
    def _t64(a):
        return tf.constant(np.ascontiguousarray(a, dtype=np.float64),
                           dtype=tf.float64)

    def outputs(self, X):
        return np.asarray(self._fwd(self._t64(X)), dtype=np.float64)

    def jacobian(self, X):
        return np.asarray(self._jac(self._t64(X)), dtype=np.float64)

    def hessian(self, X, lam):
        return np.asarray(self._hess(self._t64(X), self._t64(lam)),
                          dtype=np.float64)


def make_evaluator(name):
    return TFEvaluator(to_float64(NETS[name]))


EVALUATORS = {act: make_evaluator(act) for act in ACTIVATIONS}

# Sanity: batched float64 evaluation must agree with the float32 Keras model.
for act in ACTIVATIONS:
    got = EVALUATORS[act].outputs(X_check)
    d = np.abs(got - NETS[act].predict(X_check, verbose=0)).max()
    print(f"{act:5s}  max |float64 graph - float32 keras| on {len(X_check)} points = "
          f"{d:.2e}   (all finite: {np.isfinite(got).all()})")

### Verify the derivatives before trusting them

A wrong Jacobian sign in a grey box does not raise; it just makes IPOPT converge
somewhere wrong, or not at all. Central finite differences settle the question.

In [ ]:
def fd_jacobian(fn, X, h=1e-6):
    """Central-difference Jacobian of a batched function, shape (N, n_out, n_in)."""
    N = X.shape[0]
    J = np.zeros((N, N_OUT, N_IN))
    for j in range(N_IN):
        e = np.zeros(N_IN); e[j] = h
        J[:, :, j] = (fn(X + e) - fn(X - e)) / (2 * h)
    return J


def fd_hessian(fn_grad, X, lam, h=1e-6):
    """Central-difference Hessian of the lambda-weighted output."""
    N = X.shape[0]
    H = np.zeros((N, N_IN, N_IN))
    for j in range(N_IN):
        e = np.zeros(N_IN); e[j] = h
        gp = np.einsum("noi,no->ni", fn_grad(X + e), lam)
        gm = np.einsum("noi,no->ni", fn_grad(X - e), lam)
        H[:, :, j] = (gp - gm) / (2 * h)
    return H


rng = np.random.default_rng(SEED)
NFD = 8
_c = {"X": rng.uniform(1.0, X_BOUNDS[1], NFD),
      "V": rng.uniform(*V_BOUNDS, NFD),
      "S": rng.uniform(*S_BOUNDS, NFD),
      "U": rng.uniform(*U_BOUNDS, NFD)}
Xc = np.column_stack([_c[nm] for nm in INPUT_ORDER])
lam = rng.normal(size=(NFD, N_OUT))


def rel_err(a, b):
    return float(np.abs(a - b).max() / max(np.abs(b).max(), 1e-30))


print("TensorFlow AD vs central finite differences\n")
print(f"{'':7s}{'jacobian':>22s}{'hessian':>22s}")
print(f"{'':7s}{'max abs':>11s}{'max rel':>11s}{'max abs':>11s}{'max rel':>11s}")
for act in ACTIVATIONS:
    ev = EVALUATORS[act]
    J, Jfd = ev.jacobian(Xc), fd_jacobian(ev.outputs, Xc)
    H, Hfd = ev.hessian(Xc, lam), fd_hessian(ev.jacobian, Xc, lam)
    print(f"{act:5s}  {np.abs(J - Jfd).max():11.2e}{rel_err(J, Jfd):11.2e}"
          f"{np.abs(H - Hfd).max():11.2e}{rel_err(H, Hfd):11.2e}")

print()
print("relu's second derivative is zero almost everywhere, so its Hessian check is")
print("trivially satisfied. The tanh row is the one that tests the chain rule.")

In [ ]:
class NODEGreyBox(ExternalGreyBoxModel):
    """dS/dt = NN(X, V, S, U) at every collocation point, as one grey box.

    Flattened point-major in INPUT_ORDER / OUTPUT_ORDER, e.g. for the
    default lists:
        inputs  = [X_0, V_0, S_0, U_0, X_1, V_1, S_1, U_1, ...]
        outputs = [dSdt_0, dSdt_1, ...]

    Hessian-capable. PyNumero detects that purely from the existence of
    evaluate_hessian_outputs (hasattr, not a flag), which also obliges us to
    override set_output_constraint_multipliers - see the note on that method.
    """

    def __init__(self, evaluator, n_points, init=None):
        self.ev = evaluator
        self.N = int(n_points)
        self.init = init                  # trajectory to start from, or None
        self._x = np.zeros(self.N * N_IN, dtype=np.float64)
        self._lam = np.zeros(self.N * N_OUT, dtype=np.float64)

        # Constant Jacobian sparsity, built once: PyNumero caches the
        # structure and asserts it is identical on every call. Dense
        # per-point blocks, row = i*N_OUT + o, col = i*N_IN + j.
        i = np.repeat(np.arange(self.N), N_OUT * N_IN)
        o = np.tile(np.repeat(np.arange(N_OUT), N_IN), self.N)
        j = np.tile(np.arange(N_IN), self.N * N_OUT)
        self._jac_rows = i * N_OUT + o
        self._jac_cols = i * N_IN + j

        # Constant Hessian sparsity: lower-triangular dense blocks, one per
        # point. PyNumero rejects any entry with row < col.
        a, b = np.tril_indices(N_IN)
        pt = np.repeat(np.arange(self.N), a.size)
        self._h_a, self._h_b = a, b
        self._hess_rows = pt * N_IN + np.tile(a, self.N)
        self._hess_cols = pt * N_IN + np.tile(b, self.N)

    # --- naming ------------------------------------------------------
    def input_names(self):
        return [f"{nm}_{i}" for i in range(self.N) for nm in INPUT_ORDER]

    def output_names(self):
        return [f"{nm}_{i}" for i in range(self.N) for nm in OUTPUT_ORDER]

    # --- construction hook -------------------------------------------
    def finalize_block_construction(self, pyomo_block):
        """Bounds and starting values for the grey-box variables.

        Without this every grey-box input starts at 0.0 with no bounds,
        inconsistent with the DAE variables they are linked to, and V = 0 is
        exactly the evaluation that blows up. The outputs are warm-started
        with a forward pass so the output constraints begin (near) feasible.
        """
        x0 = np.zeros((self.N, N_IN))
        for i in range(self.N):
            for j, nm in enumerate(INPUT_ORDER):
                x0[i, j] = (float(self.init[nm][i]) if self.init is not None
                            else VAR_INIT[nm])
                v = pyomo_block.inputs[f"{nm}_{i}"]
                v.setlb(VAR_BOUNDS[nm][0])
                v.setub(VAR_BOUNDS[nm][1])
                v.value = x0[i, j]

        y0 = self.ev.outputs(x0)
        for i in range(self.N):
            for o, nm in enumerate(OUTPUT_ORDER):
                pyomo_block.outputs[f"{nm}_{i}"].value = float(y0[i, o])

    # --- state -------------------------------------------------------
    def set_input_values(self, x):
        self._x[:] = np.asarray(x, dtype=np.float64).reshape(-1)

    def set_output_constraint_multipliers(self, lam):
        """Required override, not an optional refinement.

        The base implementation guards on a legacy attribute name that does
        not exist any more, so it returns without storing anything. Inherit
        it and evaluate_hessian_outputs below assembles from stale zeros on
        every iteration, silently.
        """
        self._lam[:] = np.asarray(lam, dtype=np.float64).reshape(-1)

    def _X(self):
        return self._x.reshape(self.N, N_IN)

    # --- evaluation --------------------------------------------------
    def evaluate_outputs(self):
        return self.ev.outputs(self._X()).reshape(-1)

    def evaluate_jacobian_outputs(self):
        J = self.ev.jacobian(self._X())                    # (N, n_out, n_in)
        return sp.coo_matrix(
            (J.reshape(-1), (self._jac_rows, self._jac_cols)),
            shape=(self.N * N_OUT, self.N * N_IN),
        )

    def evaluate_hessian_outputs(self):
        """Lower triangle of sum_o lam_o * d2(out_o)/dx2, block diagonal.

        Positive sign: PyNumero writes the residual as f(u) - o = 0, so the
        Lagrangian contribution in the inputs is lam^T f(u).
        """
        lam = self._lam.reshape(self.N, N_OUT)
        H = self.ev.hessian(self._X(), lam)                # (N, n_in, n_in)
        vals = H[:, self._h_a, self._h_b].reshape(-1)
        n = self.N * N_IN
        return sp.coo_matrix(
            (vals, (self._hess_rows, self._hess_cols)), shape=(n, n)
        )


# PyNumero decides Hessian support by hasattr, so deleting either method
# below silently downgrades the solve rather than raising.
assert hasattr(NODEGreyBox, "evaluate_hessian_outputs")
assert "set_output_constraint_multipliers" in vars(NODEGreyBox)
print("NODEGreyBox: Hessian support detected by PyNumero =",
      hasattr(NODEGreyBox, "evaluate_hessian_outputs"))

In [ ]:
def build_efe(evaluator, nfe=NFE, ncp=NCP_REF, init=None):
    """A single grey-box block covering all collocation points."""
    m = base_ocp(nfe, ncp)
    if init is not None:
        warm_start(m, init)

    tps = sorted(m.t)
    idx = {t: i for i, t in enumerate(tps)}

    m.nn = ExternalGreyBoxBlock()
    m.nn.set_external_model(NODEGreyBox(evaluator, len(tps), init=init))

    # --- linking: DAE variables <-> grey-box inputs (by name) ---
    m.link_in = Constraint(
        m.t, range(N_IN),
        rule=lambda M, t, i: getattr(M, INPUT_ORDER[i])[t]
        == M.nn.inputs[f"{INPUT_ORDER[i]}_{idx[t]}"])

    # --- linking: grey-box output -> derivative (by name) ---
    m.link_out = Constraint(
        m.t, range(N_OUT),
        rule=lambda M, t, o: Constraint.Skip if t == 0
        else getattr(M, OUTPUT_ORDER[o])[t]
        == M.nn.outputs[f"{OUTPUT_ORDER[o]}_{idx[t]}"])
    return m


if HAVE_EFE:
    _init = REF_BY_NCP[NCP_REF]["traj"] if WARM_START else None
    _m = build_efe(EVALUATORS["tanh"], ncp=NCP_REF, init=_init)
    _gb = _m.nn.get_external_model()
    print(f"grey box at ncp={NCP_REF}: {_gb.N} points, "
          f"{len(_gb.input_names())} inputs, {len(_gb.output_names())} outputs")
    print(f"first inputs : {_gb.input_names()[:8]}")
    print(f"first outputs: {_gb.output_names()[:3]}")

    _s = SolverFactory("cyipopt")
    _s.config.options["tol"] = TOL
    _s.config.options["max_iter"] = MAX_ITER
    _s.config.options["print_level"] = 0
    _r = _s.solve(_m)
    print(f"\nsmoke solve: {_r.solver.termination_condition}, "
          f"objective {value(_m.obj):.8f}   "
          f"(mechanistic reference {REF_BY_NCP[NCP_REF]['obj']:.8f})")
else:
    print("EFE unavailable - see the environment check at the top.")

In [ ]:
# --- cost per time point: the OMLT blocks of Section 3.2 next to EFE ---
# One OMLT block covers one time point. The grey box covers all N points at
# once, so a one-point grey box measures what it adds per point. The linking
# constraints (N_IN + N_OUT per point) sit outside both and are left out.
def efe_size_per_point(name):
    """Variables / constraints / binaries one time point adds to the grey box."""
    gb = NODEGreyBox(EVALUATORS[name], n_points=1)
    n_var = len(gb.input_names()) + len(gb.output_names())
    n_con = len(gb.output_names())   # one implicit f(u) - o = 0 per output
    return n_var, n_con, 0


per_point = []
for act in ACTIVATIONS:
    for fname in ECE_FORMULATIONS:
        try:
            nv, nc, nb_ = block_size(act, fname)
        except ValueError:
            continue                  # unsupported pair, see the 3.2 table
        per_point.append({"activation": act, "formulation": fname,
                          "vars/point": nv, "cons/point": nc,
                          "binaries/point": nb_})
    nv, nc, nb_ = efe_size_per_point(act)
    per_point.append({"activation": act, "formulation": "EFE",
                      "vars/point": nv, "cons/point": nc,
                      "binaries/point": nb_})

print(f"cost per time point, {ARCH} network\n")
print(pd.DataFrame(per_point).to_string(index=False))

---
# 4 &nbsp;&nbsp;The comparison

Now we run every formulation, on both networks, across `NCP_LIST` (just `NCP_REF` when
`QUICK_TEST = True`)

* **problem size**: variables and constraints as the solver sees them
* **build time**: constructing the Pyomo model, which is *not* free for ECE
* **solve time**: median of `N_REPS` repeated solves
* **first-run time** (`t_first_s`): build + solve the first time each case runs,
  one-time costs included
* **iterations** and **seconds per iteration**: because a method that takes 10x
  longer with 10x the iterations is telling you something different from one that
  takes 10x longer per iteration
* **objective**: against the mechanistic reference at the same `ncp`
* **the recovered control and state trajectories**
* **what the recovered feed profile is worth in the true plant** (Section 4.4), which
  matters more here than in CS1 because only one equation is learned

### Three things that have to be right for the numbers to mean anything

**First run vs repeated runs.** The first time a case runs it pays one-time costs that
later runs do not, the biggest being TensorFlow tracing the EFE evaluator's graphs. So
every case gets one first run, timed on its own as `t_first_s` (build + solve), followed
by `N_REPS` repeated runs whose medians are `t_build_s` and `t_solve_s`. The two answer
different questions: `t_first_s` is what a one-off solve costs, the medians are what each
solve costs once the method is set up. Each EFE case gets a fresh evaluator, so its first
run includes its own tracing instead of reusing graphs traced by earlier cells.

Process-level costs (importing the packages, loading IPOPT and the PyNumero ASL library)
are paid once by earlier cells, before the sweep starts, so they appear in none of these
columns. A truly cold start would need a fresh Python process per case.

Python's garbage collector (the cyclic garbage collector) is run before every timed run
to remove unused objects from the system's memory. ECE models create a very large number
of Python objects, and without this a full collection lands on roughly every other run,
wherever the object count happens to cross the threshold, which makes both the medians
and `t_first_s` unreliable.

**Build time separately from solve time.** ECE constructs `N` copies of the network as
Pyomo algebra; EFE constructs one grey-box block. That construction cost is part of what
the formulation costs you, but it is not solver time, so lumping them together would
misattribute it.

**A common starting point.** With `WARM_START = True` every case at a given `ncp` starts
from the mechanistic solution at that `ncp` (Section 1 explains why). With
`WARM_START = False` every case starts from `VAR_INIT`. Either way, every formulation
starts from the same point.

### On comparing `ipopt` with `cyipopt`

Both are IPOPT. The difference is the interface:

* **`ipopt`**: Pyomo writes an `.nl` file, the solver executable reads it, AMPL's
  AD library supplies derivatives.
* **`cyipopt`**: Python calls IPOPT in-process. For an ordinary Pyomo model this
  *still* goes through an `.nl` file and ASL internally, so it mostly just swaps
  the driver. For a grey-box model it is the only option, because a callback into
  TensorFlow cannot be written into an `.nl` file.

Put simply: ECE always pays for writing and differentiating the network as algebra; EFE
moves that cost into Python/TensorFlow callbacks. Running both solver backends on the
ECE rows is what lets you separate the two effects instead of guessing.

In [ ]:
# Per-process: two kernels (or a stale one) sharing this path fight over the
# file, and an IPOPT that cannot open its output_file aborts the solve.
IPOPT_LOG = os.path.join(tempfile.gettempdir(),
                         f"cs3_ipopt_iters_{os.getpid()}.log")
ITER_RE = re.compile(r"Number of Iterations\.*:\s*(\d+)")


def solver_options():
    """Options common to both backends.

    output_file/file_print_level are how we recover the iteration count:
    neither Pyomo interface populates results.solver.statistics reliably,
    but IPOPT itself will write its log wherever we ask.
    """
    return {
        "tol": TOL,
        "max_iter": MAX_ITER,
        "print_level": 0,
        "output_file": IPOPT_LOG,
        "file_print_level": 4,
    }


def make_solver(solver_name, opts):
    """cyipopt takes options through .config.options, the .nl interface
    through .options. Assigning a whole dict to config.options silently
    drops every entry, which is a memorable way to lose max_iter and spend
    an afternoon confused."""
    s = SolverFactory(solver_name)
    target = s.config.options if solver_name == "cyipopt" else s.options
    for key, val in opts.items():
        target[key] = val
    return s


def read_iterations():
    try:
        with open(IPOPT_LOG) as fh:
            match = ITER_RE.search(fh.read())
        return int(match.group(1)) if match else np.nan
    except OSError:
        return np.nan


def problem_size(m):
    """Variables / constraints, including any grey-box block.

    ExternalGreyBoxBlock carries its own ctype, so the ordinary Var traversal
    (descend_into=True descends into Block) may or may not reach its
    inputs/outputs depending on the Pyomo version. Deduplicating by id() is
    correct either way. The block also contributes one implicit output
    constraint f(u) - o = 0 per output, which no traversal will show.
    """
    seen = {}

    def collect(container):
        for v in container.component_data_objects(Var, active=True,
                                                  descend_into=True):
            seen[id(v)] = v

    collect(m)
    n_gb_con = 0
    for blk in m.component_data_objects(ExternalGreyBoxBlock, active=True,
                                        descend_into=True):
        collect(blk)
        n_gb_con += len(blk.get_external_model().output_names())

    n_con = sum(1 for _ in m.component_data_objects(Constraint, active=True,
                                                    descend_into=True))
    return len(seen), n_con + n_gb_con


def method_label(formulation, network):
    """Display name used in tables and plot legends, e.g. "ECE-FS (tanh)"."""
    return f"{formulation} ({network})"


print("harness ready")

In [ ]:
def run_case(case, n_reps=N_REPS):
    """Time one first run (build + solve), then n_reps repeated runs.

    The first run carries any one-time cost this case is the first to pay
    (e.g. TF tracing) and is reported on its own as t_first_s. Every run,
    first or repeated, builds a fresh model and solves it from the same
    starting point. Python's garbage collector is run before every timed
    run, so a collection pause triggered by earlier runs does not land on
    whichever run happens to be next. Returns a row dict plus the trajectory
    of the last solved model.
    Unsupported combinations raise inside build(); the exception is recorded
    rather than propagated. The same applies to a solver that
    dies mid-sweep.
    """
    row = {
        "ncp": case["ncp"],
        "formulation": case["formulation"],
        "activation": case["activation"],
        "solver": case["solver"],
        "hessian": case["hessian"],
        "note": case.get("note", ""),
    }
    opts = solver_options()

    # --- first run, timed on its own: includes one-time costs ---
    try:
        gc.collect()
        t0 = time.perf_counter()
        m_first = case["build"]()
        make_solver(case["solver"], opts).solve(m_first)
        t_first = time.perf_counter() - t0
        del m_first
    except Exception as exc:
        row["status"] = f"FAILED: {type(exc).__name__}: {exc}"
        return row, None

    # --- repeated runs, each from a fresh model ---
    t_build, t_solve = [], []
    m = res = None
    try:
        for _ in range(n_reps):
            m = None              # drop the previous model, then collect
            gc.collect()
            t0 = time.perf_counter()
            m = case["build"]()
            t_build.append(time.perf_counter() - t0)

            solver = make_solver(case["solver"], opts)
            t0 = time.perf_counter()
            res = solver.solve(m)
            t_solve.append(time.perf_counter() - t0)
    except Exception as exc:
        row["status"] = f"FAILED: {type(exc).__name__}: {exc}"
        return row, None

    n_var, n_con = problem_size(m)
    iters = read_iterations()
    t_med = float(np.median(t_solve))

    row.update({
        "variables": n_var,
        "constraints": n_con,
        "t_build_s": round(float(np.median(t_build)), 4),
        "t_solve_s": round(t_med, 4),
        "t_solve_sd": round(float(np.std(t_solve)), 4),
        "t_first_s": round(t_first, 4),
        "iters": iters,
        "ms_per_iter": (round(1000 * t_med / iters, 3)
                        if iters and np.isfinite(iters) and iters > 0
                        else np.nan),
        "objective": round(float(value(m.obj)), 8),
        "status": str(res.solver.termination_condition),
    })
    return row, trajectories(m)


def make_cases(ncp):
    """Every (formulation, activation, solver) combination to run.

    ReLU appears only with ECE-CMP, the one formulation that requires a
    piecewise-linear activation. EFE is a smooth-NLP method and is paired
    with tanh, where the network's second derivatives exist and are nonzero.
    Every case uses exact Hessians, and every case at this ncp starts from
    the same point.
    """
    init = REF_BY_NCP[ncp]["traj"] if WARM_START else None
    cases = []

    allowed_ece = [
        ("ECE-FS", "tanh"),
        ("ECE-RS", "tanh"),
        ("ECE-CMP", "relu"),
    ]

    for fname, act in allowed_ece:
        for solver in ("ipopt", "cyipopt"):
            cases.append({
                "ncp": ncp, "formulation": fname, "activation": act,
                "solver": solver, "hessian": "exact", "note": "",
                "build": (lambda a=act, f=fname, c=ncp, i=init:
                          build_ece(a, f, ncp=c, init=i)),
            })

    if HAVE_EFE:
        # A fresh evaluator per case: its tf.functions have never been traced,
        # so this case's first run pays its own tracing rather than reusing
        # graphs traced by the checks earlier in the notebook.
        ev = make_evaluator("tanh")
        cases.append({
            "ncp": ncp, "formulation": "EFE", "activation": "tanh",
            "solver": "cyipopt", "hessian": "exact", "note": "",
            "build": (lambda c=ncp, e=ev, i=init: build_efe(e, ncp=c, init=i)),
        })

    return cases


n_cases = len(make_cases(NCP_LIST[0]))
print(f"{n_cases} cases per ncp x {len(NCP_LIST)} ncp values "
      f"= {n_cases * len(NCP_LIST)} runs")
print(f"each run = 1 first run (t_first_s) + {N_REPS} repeated runs (medians)")
print(f"starting point: {'mechanistic solution (WARM_START)' if WARM_START else 'VAR_INIT'}")

This is the long cell. Every case is solved `1 + N_REPS` times at every `ncp`, and most
of the time goes to the ReLU complementarity cases, the `cyipopt` one in particular,
which is itself part of the result. At the default settings, expect it to take on the
order of half an hour to an hour on a laptop CPU.

In [ ]:
results, TRAJ = [], {}
t_start = time.perf_counter()

for ncp in NCP_LIST:
    print(f"\n=== ncp = {ncp} " + "=" * 52)
    for case in make_cases(ncp):
        tag = (f"{case['formulation']:8s} {case['activation']:5s} "
               f"{case['solver']:8s} {case['hessian']:6s}")
        row, traj = run_case(case)
        results.append(row)
        if traj is not None:
            TRAJ[(ncp, case["formulation"], case["activation"],
                  case["solver"], case["hessian"])] = traj

        if row["status"].startswith("FAILED"):
            print(f"  {tag}  {row['status'][:78]}")
        else:
            print(f"  {tag}  t={row['t_solve_s']:7.3f}s  "
                  f"first={row['t_first_s']:7.3f}s  "
                  f"iters={row['iters']:>4}  obj={row['objective']:.6f}  "
                  f"{row['status']}", flush=True)

RESULTS = pd.DataFrame(results)
print(f"\ntotal wall time {time.perf_counter() - t_start:.1f}s")

## 4.1 &nbsp;Solve time and problem size

Every row here is IPOPT. The `solver` column says how the problem *reached* it, and
the two routes are not interchangeable:

* **`ipopt`** writes the model to an AMPL `.nl` file and runs the `ipopt` executable
  as a subprocess. Every function and derivative evaluation happens in compiled C
  through the ASL; Python is out of the loop once the file is written.
* **`cyipopt`** links IPOPT in-process. PyNumero assembles the NLP in Python, and the
  solver calls back across the Python boundary for residuals and derivatives on every
  iteration.

EFE can only use `cyipopt`. The `.nl` format is purely algebraic and has no way to
encode "call this TensorFlow function", so a grey box exists only on the in-process
path. ECE is run on both: the two routes agreeing is a check on the wiring, and it
supplies the `cyipopt` row that EFE can actually be compared against.

That second reason matters, because the routes differ in cost (see `ms_per_iter`). The
in-process penalty grows with problem size, since every iteration ships the full
Jacobian through Python instead of staying inside ASL. A formulation comparison drawn
from the `cyipopt` rows is like-for-like; one that mixes `.nl` ECE against in-process
EFE is partly measuring the interface.

The two routes need not even run the same IPOPT. Section 0 printed the version behind
each; where this notebook was written, the IDAES `ipopt` executable is IPOPT 3.13 with
the HSL MA27 linear solver, and `cyipopt` links IPOPT 3.14 with MUMPS. Different linear
solvers take slightly different steps, so iteration counts can differ between the two
rows of one formulation, and on the complementarity problem they can stop at slightly
different points.

In [ ]:
ok = RESULTS[~RESULTS["status"].str.startswith("FAILED")].copy()
ok["method"] = [method_label(f, a)
                for f, a in zip(ok["formulation"], ok["activation"])]
# Only rows that converged to an optimum are evidence about where the
# optimum is; a run that hit max_iter just stopped early.
ok["trustworthy"] = ok["status"] == "optimal"

cols = ["ncp", "method", "solver", "hessian", "variables", "constraints",
        "t_build_s", "t_solve_s", "t_solve_sd", "t_first_s", "iters",
        "ms_per_iter", "objective", "status"]

for ncp in NCP_LIST:
    sub = ok[ok["ncp"] == ncp][cols].sort_values(["method", "solver"])
    print(f"\n{'=' * 118}\nncp = {ncp}    "
          f"({NFE * ncp + 1} time points)    "
          f"mechanistic reference -P(tf)V(tf) = {REF_BY_NCP[ncp]['obj']:.8f}\n"
          f"{'=' * 118}")
    print(sub.to_string(index=False))

fails = RESULTS[RESULTS["status"].str.startswith("FAILED")]
if len(fails):
    print(f"\n{'=' * 118}\nUnsupported combinations (the empirical support matrix)"
          f"\n{'=' * 118}")
    seen = fails.drop_duplicates(subset=["formulation", "activation"])
    for _, r in seen.iterrows():
        print(f"  {r['formulation']:8s} + {r['activation']:5s}  ->  "
              f"{r['status'].split(': ', 1)[-1]}")

In [ ]:
# Problem size does not depend on the solver backend, so one row per
# (ncp, method) is enough.
size = ok.drop_duplicates(subset=["ncp", "method"])

piv_v = size.pivot(index="ncp", columns="method", values="variables")
piv_c = size.pivot(index="ncp", columns="method", values="constraints")

# The mechanistic model is the floor: what the OCP costs with no network at all.
for ncp in NCP_LIST:
    piv_v.loc[ncp, "mechanistic"], piv_c.loc[ncp, "mechanistic"] = \
        problem_size(mechanistic_model(ncp=ncp))

# Int64 (nullable) rather than int: a method that failed at one ncp leaves a gap.
print("VARIABLES\n"); print(piv_v.astype("Int64").to_string())
print("\nCONSTRAINTS\n"); print(piv_c.astype("Int64").to_string())

In [ ]:
# --- build vs solve, at the largest ncp ---
# Every (method, solver) row, so each ECE method shows both the .nl (ipopt) and
# the in-process (cyipopt) route, and EFE is compared like-for-like with the
# cyipopt rows. Listed alphabetically from the top.
ncp_max = max(NCP_LIST)
bar = (ok[ok["ncp"] == ncp_max]
       .sort_values(["method", "solver"], ascending=False))

# Build and solve side by side, not stacked: on a log axis a stacked segment's
# length does not show its value, and a very short build (EFE) falls off the
# left edge entirely. Note t_solve_s includes Pyomo writing the model out for
# IPOPT, which is larger for the ECE formulations.
fig, ax = plt.subplots(figsize=(9, 0.9 * len(bar) + 1.2))
y = np.arange(len(bar))
h = 0.38
b_build = ax.barh(y + h / 2, bar["t_build_s"], height=h, color=BUILD_COLOR,
                  label="build (Pyomo construction)")
b_solve = ax.barh(y - h / 2, bar["t_solve_s"], height=h, color=SOLVE_COLOR,
                  label="solve (IPOPT)")
for container in (b_build, b_solve):
    ax.bar_label(container, fmt="%.3g s", padding=3, fontsize=7)
t_all = np.concatenate([bar["t_build_s"], bar["t_solve_s"]])
ax.set_xlim(t_all.min() / 2, t_all.max() * 4)   # room for the value labels
ax.set_yticks(y)
ax.set_yticklabels([f"{m}\n[{s}]" for m, s in
                    zip(bar["method"], bar["solver"])], fontsize=8)
ax.set_xlabel("seconds")
ax.set_xscale("log")
ax.set_title(f"Build vs solve at ncp={ncp_max} ({NFE * ncp_max + 1} time points)")
ax.legend()
fig.tight_layout()
fig.savefig("CS3_naive_tutorial_build_vs_solve.png")
plt.show()

In [ ]:
# --- first run vs repeated runs, next to the build/solve chart above ---
# t_first_s is the first build + solve of each case, one-time costs included.
# setup_s = t_first_s - (t_build_s + t_solve_s): what the first run cost beyond
# a typical repeated run, mostly TensorFlow tracing for EFE. Small values
# (a few hundredths of a second, either sign) are timing noise.
first = ok[["ncp", "method", "solver", "t_build_s", "t_solve_s",
            "t_first_s"]].copy()
first["build+solve_s"] = first["t_build_s"] + first["t_solve_s"]
first["setup_s"] = first["t_first_s"] - first["build+solve_s"]
first = first[["ncp", "method", "solver", "t_build_s", "t_solve_s",
               "build+solve_s", "t_first_s", "setup_s"]]

for ncp in NCP_LIST:
    sub = (first[first["ncp"] == ncp]
           .drop(columns="ncp")
           .sort_values("t_first_s")
           .round(4))
    print(f"\nncp = {ncp}   ({NFE * ncp + 1} time points)")
    print(sub.to_string(index=False))

## 4.2 &nbsp;Control and state trajectories

Every formulation is a different encoding of the same mathematics, so every formulation
that is wired up correctly and reaches the same local optimum must return the **same
trajectories**, up to surrogate error, which differs between the `tanh` and `relu`
networks but not between formulations sharing a network. Section 4.3 checks whether
they do reach the same local optimum.

The dashed black line is the mechanistic solution at the same `ncp`. Where formulations
coincide, each later one is drawn thinner on top of the one before, so every formulation
stays visible as an outline. $S$ is drawn on a symmetric-log axis, linear below
$10^{-3}$ g/L and logarithmic above, so the feed peak and the low-$S$ production phase
are both visible.

In [ ]:
def representative(ncp, formulation, activation):
    """One trajectory per (formulation, activation): backends agree, so we
    only need one. Prefer the .nl backend for determinism."""
    for solver in ("ipopt", "cyipopt"):
        key = (ncp, formulation, activation, solver, "exact")
        if key in TRAJ:
            return TRAJ[key]
    return None


combos = sorted({(r["formulation"], r["activation"]) for r in results
                 if not r["status"].startswith("FAILED")})

fig, axes = plt.subplots(1, len(NCP_LIST), figsize=(5 * len(NCP_LIST), 4),
                         sharey=True)
axes = np.atleast_1d(axes)

for a, ncp in zip(axes, NCP_LIST):
    a.plot(REF_BY_NCP[ncp]["traj"]["t"], REF_BY_NCP[ncp]["traj"]["U"], **REF_KW)
    # Formulations that share a network give identical trajectories, so draw
    # each successive one thinner and smaller on top of the last (layer_kw).
    for k_, (fname, act) in enumerate(combos):
        tr = representative(ncp, fname, act)
        if tr is None:
            continue
        meth = method_label(fname, act)
        a.plot(tr["t"], tr["U"], "-", label=meth,
               **method_style(meth), **layer_kw(k_))
    a.axhline(U_BOUNDS[0], **BOUND_KW)
    a.axhline(U_BOUNDS[1], **BOUND_KW)
    a.set_xlabel("time (h)")
    a.set_title(f"ncp = {ncp}   ({NFE * ncp + 1} points)")

axes[0].set_ylabel(ylabel("U"))
axes[-1].legend(loc="upper right")
fig.suptitle("Optimal feed profile by formulation", y=1.02)
fig.tight_layout()
fig.savefig("CS3_naive_tutorial_control_trajectories.png")
plt.show()

In [ ]:
# squeeze=False keeps axes 2-D (n_states x n_ncp) even for a single ncp.
fig, axes = plt.subplots(len(STATE_ORDER), len(NCP_LIST),
                         figsize=(5 * len(NCP_LIST), 2.9 * len(STATE_ORDER)),
                         sharey="row", squeeze=False)

for col, ncp in enumerate(NCP_LIST):
    ref = REF_BY_NCP[ncp]["traj"]
    for r, sname in enumerate(STATE_ORDER):
        a = axes[r, col]
        a.plot(ref["t"], ref[sname], **REF_KW)
        for k_, (fname, act) in enumerate(combos):
            tr = representative(ncp, fname, act)
            if tr is None:
                continue
            meth = method_label(fname, act)
            a.plot(tr["t"], tr[sname], "-", label=meth,
                   **method_style(meth), **layer_kw(k_, lw0=2.8, ms0=6.5))
        if sname == "S":
            a.set_yscale("symlog", linthresh=1e-3)
        a.set_ylabel(ylabel(sname))
        if r == 0:
            a.set_title(f"ncp = {ncp}")
        if r == len(STATE_ORDER) - 1:
            a.set_xlabel("time (h)")

axes[0, -1].legend(fontsize=7)
fig.suptitle("State trajectories by formulation", y=1.0)
fig.tight_layout()
fig.savefig("CS3_naive_tutorial_state_trajectories.png")
plt.show()

## 4.3 &nbsp;Do the formulations agree?


1. **Within one network, do all formulations return the same optimum?**
   In CS1 they did, to within the solver tolerance. A nonzero spread points to a bug
   inside one formulation (a wrong grey-box derivative, a bounds or linking mistake on
   one path) or to formulations landing in different local optima of this nonconvex
   problem. Either way it is *not* a property of the network. Section 1 showed that
   this problem has more than one local optimum, so here a mismatch is not
   automatically a bug, and the cell after this one tells the two causes apart.
2. **How far is each network's optimum from the mechanistic one?**
   That gap is surrogate error, and it is the price of using a NODE at all.

The objective here is of order 100, so agreement is judged relative to its size.

In [ ]:
# Only converged rows are evidence about the optimum. Including a
# maxIterations row would report a mismatch that is really just a solve
# that stopped early.
trust = ok[ok["trustworthy"]]
AGREE_RTOL = 1e-6          # relative spread counted as agreement

print("1) Agreement across formulations sharing the same network")
print("   (nonzero spread => formulation-specific bug or a different local optimum)\n")

agree = []
for (ncp, act), grp in trust.groupby(["ncp", "activation"]):
    n_form = grp["formulation"].nunique()
    spread = grp["objective"].max() - grp["objective"].min()
    rel = spread / abs(grp["objective"].median())
    if n_form < 2:
        verdict = "only 1 formulation converged - nothing to compare"
    elif rel < AGREE_RTOL:
        verdict = "OK"
    else:
        verdict = "MISMATCH - formulation bug or different local optimum"
    agree.append({
        "ncp": ncp, "activation": act,
        "formulations": n_form, "rows": len(grp),
        "objective": grp["objective"].median(),
        "spread": spread, "relative spread": rel, "verdict": verdict,
    })
print(pd.DataFrame(agree).to_string(index=False))

print("\n\n2) Surrogate error: NODE optimum vs mechanistic optimum\n")
err = []
for (ncp, act), grp in trust.groupby(["ncp", "activation"]):
    ref_obj = REF_BY_NCP[ncp]["obj"]
    node_obj = grp["objective"].median()
    err.append({
        "ncp": ncp, "activation": act,
        "mechanistic -P*V": round(ref_obj, 6),
        "NODE -P*V": round(node_obj, 6),
        "abs error": round(abs(node_obj - ref_obj), 6),
        "rel error %": round(100 * abs(node_obj - ref_obj) / abs(ref_obj), 3),
        # The objective is minimised, so a NODE value below the reference
        # promises more penicillin than the true model can make.
        "optimistic?": "yes" if node_obj < ref_obj else "no",
    })
print(pd.DataFrame(err).to_string(index=False))

not_conv = ok[ok["status"] != "optimal"]
if len(not_conv):
    print(f"\n   ({len(not_conv)} run(s) did not converge and are excluded "
          f"above:")
    for _, r in not_conv.iterrows():
        print(f"      ncp={r['ncp']} {r['method']} [{r['solver']}, "
              f"{r['hessian']}] -> {r['status']} after {r['iters']} iters")
    print("   )")

### Telling a bug from a different local optimum

The two causes of a spread in table 1 can be separated without guessing. Take the
optimal feed $U^\star(t)$ that one formulation found, fix it in every other
formulation's model, and solve what is left. With $U$ fixed at every time point there
are no degrees of freedom: each model only computes the states and the objective that
this feed produces. If the formulations encode the same model, they must report the same
objective for the same feed, whichever formulation the feed came from.

So in the table below, a difference **along a row** (same feed, different models) would
be a wiring error. A difference **between rows** (different feeds) is the optimizer
finding different local optima. The check uses the `tanh` formulations at `NCP_REF`,
the network they all share.

In [ ]:
def tanh_builders(ncp):
    """(build function taking a warm-start trajectory, solver) per tanh formulation."""
    b = {"ECE-FS": (lambda i: build_ece("tanh", "ECE-FS", ncp=ncp, init=i), "ipopt"),
         "ECE-RS": (lambda i: build_ece("tanh", "ECE-RS", ncp=ncp, init=i), "ipopt")}
    if HAVE_EFE:
        b["EFE"] = (lambda i: build_efe(make_evaluator("tanh"), ncp=ncp, init=i),
                    "cyipopt")
    return b


def evaluate_fixed_control(build, solver_name, traj):
    """Fix U at every time point to traj's values and solve the square system
    that is left. traj is also the starting point."""
    m = build(traj)
    for i, t in enumerate(sorted(m.t)):
        m.U[t].fix(float(traj["U"][i]))
    res = make_solver(solver_name, solver_options()).solve(m)
    return float(value(m.obj)), str(res.solver.termination_condition)


builders = tanh_builders(NCP_REF)
feeds = {f: representative(NCP_REF, f, "tanh") for f in builders}
feeds = {f: tr for f, tr in feeds.items() if tr is not None}

first = next(iter(feeds))              # feeds are compared with this one
xrows = []
for src, traj in feeds.items():
    row = {"U* from": method_label(src, "tanh")}
    for f, (build, solver_name) in builders.items():
        obj, status = evaluate_fixed_control(build, solver_name, traj)
        row[f"in {f}"] = obj if status == "optimal" else np.nan
    vals = np.array([row[f"in {f}"] for f in builders])
    row["spread along row (rel)"] = float(np.ptp(vals) / abs(np.mean(vals)))
    row[f"max |U - U({first})| (L/h)"] = float(np.abs(traj["U"] - feeds[first]["U"]).max())
    xrows.append(row)

xdf = pd.DataFrame(xrows)
print(f"objective -P(tf)V(tf) of a FIXED feed, ncp={NCP_REF}\n")
print(xdf.to_string(index=False, float_format=lambda v: f"{v:.8g}"))

row_spread = xdf["spread along row (rel)"].max()
print()
if row_spread < 1e-5:
    print(f"Along every row the models agree to {row_spread:.1e} (relative): same feed,")
    print("same objective, so the embeddings compute the same model. Differences")
    print("between rows come from different feeds, i.e. different local optima.")
else:
    print(f"Rows disagree by up to {row_spread:.1e} (relative): the same feed gives")
    print("different objectives in different formulations. Check the wiring.")

Each row agrees to several digits across the models, so the embeddings compute the
same model. Where two rows differ, the feeds behind them differ too (last column, and
the curves in 4.2): those formulations stopped at different local optima of the hybrid
model, whose objectives can lie very close together. That is a property of the problem,
not of any embedding, and choosing between the local optima (by multistart, say) is
outside the scope of this notebook.

## 4.4 &nbsp;What does the NODE-optimal feed actually achieve?

The objective in 4.3 is the optimum *of the hybrid model*. If the surrogate is wrong
about $\dot S$, the hybrid model is wrong about how much substrate is in the vessel,
and through $\mu$ and $\rho$ about how much biomass and penicillin it makes. It can then
report an objective the real plant cannot deliver.

So take the recovered feed profile $U^\star(t)$ from each solve, hold it fixed, and
integrate the **true** four-state ODE system with it (`scipy.integrate.solve_ivp`). The
resulting $-P(t_f)V(t_f)$ is what that feed policy is actually worth.

| | meaning |
|---|---|
| **model objective** | what the optimizer believed |
| **true objective** | what the same control actually delivers |
| **optimality gap** | true objective vs the mechanistic optimum, re-simulated the same way (positive is worse) |

The mechanistic control goes through the identical pipeline, so its own row absorbs the
difference between the collocation solution and `solve_ivp`, and the comparison is
like-for-like.

### Rebuilding $U(t)$ between collocation points

`pyomo.dae` stores $U$ only at the collocation points, so the feed has to be rebuilt as
a function of time before it can be simulated. Inside each finite element of length $h$,
the collocation model advances every state with the element's quadrature weights $w_j$:
$x(\text{end}) = x(\text{start}) + h\sum_j w_j f_j$, with $f_j$ the right-hand side at
collocation point $j$ (for Radau with `ncp=2` the weights are $3/4$ and $1/4$). So $U$ is
rebuilt as a step profile that holds each collocation value $U_j$ for a share $w_j$ of
the element, in order. That feeds exactly the substrate the model fed in every element,
and it stays inside $0 \le U \le 50$. A polynomial through the collocation points would
also feed the right amount, but it can swing below zero between the points, and a
negative feed is not something the plant can do. Since $\dot V = U/S_F$ is mechanistic in
every model, $V(t_f)$ in the model and in the simulation must agree, and the table
checks it.

### A small negative gap is not a win

A NODE row can come out slightly **better** than the mechanistic one. That does not mean
the surrogate beat the physics it was trained on. The reference is the optimum of the
*collocation discretization at this `nfe`*, not of the continuous problem, and its
control is only as good as that grid. Re-simulated exactly it loses a little, and a
slightly different control can do better. Raising `NFE` should shrink such a gap toward
zero.

In [ ]:
def quadrature_weights(tau):
    """w_j = integral over [0, 1] of the Lagrange basis polynomial through the
    collocation points tau (in element coordinates). Radau, ncp=2: [3/4, 1/4]."""
    w = []
    for j in range(len(tau)):
        others = np.delete(tau, j)
        basis = np.poly(others) / np.prod(tau[j] - others)
        integral = np.polyint(basis)
        w.append(np.polyval(integral, 1.0) - np.polyval(integral, 0.0))
    return np.array(w)


def control_steps(traj, nfe=NFE):
    """The feed as a step profile: (breakpoints, values). In every element,
    collocation value U_j is held for a share w_j of the element (see the
    markdown above)."""
    ts, us = np.asarray(traj["t"], float), np.asarray(traj["U"], float)
    edges = np.linspace(0.0, T_FINAL, nfe + 1)
    breaks, values = [0.0], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (ts > lo + 1e-9) & (ts <= hi + 1e-9)
        for wj, uj in zip(quadrature_weights((ts[sel] - lo) / (hi - lo)), us[sel]):
            breaks.append(breaks[-1] + wj * (hi - lo))
            values.append(uj)
    return np.array(breaks), np.array(values)


def true_rhs(y, U):
    """The TRUE four-state right-hand side, for a given feed U."""
    V, X, P, S = y
    V, X, S = max(V, 1e-9), max(X, 0.0), max(S, 0.0)
    den = KX * X + S
    mu = MU_MAX * S / den if den > 0 else 0.0
    rho = RHO_MAX * S / (KP + S * (1 + S / KIN))
    return [U / SF,
            mu * X - (X / (SF * V)) * U,
            rho * X - KDEG * P - (P / (SF * V)) * U,
            (-mu * X / YXS - rho * X / YPS - (MS * S / (KM + S)) * X
             + (1 - S / SF) * (U / V))]


def simulate_true(traj, pts_per_step=12):
    """Integrate the true dynamics under traj's feed, one step at a time, so
    the integrator never steps across a jump in U."""
    breaks, values = control_steps(traj)
    y, ok = np.array([V0, X0, P0, S0], float), True
    T, Y, U = [0.0], [y], [values[0]]
    for t0, t1, u in zip(breaks[:-1], breaks[1:], values):
        sol = solve_ivp(lambda t, yy: true_rhs(yy, u), (t0, t1), y,
                        t_eval=np.linspace(t0, t1, pts_per_step),
                        method="LSODA", rtol=1e-9, atol=1e-11)
        ok = ok and bool(sol.success)
        y = sol.y[:, -1]
        T.extend(sol.t[1:]); Y.extend(sol.y[:, 1:].T); U.extend([u] * (len(sol.t) - 1))
    V, X, P, S = np.array(Y).T
    return {"t": np.array(T), "V": V, "X": X, "P": P, "S": S, "U": np.array(U),
            "obj": float(-P[-1] * V[-1]), "ok": ok}


def playback_row(ncp, method, model_obj, traj, sim, true_ref):
    return {
        "ncp": ncp, "method": method,
        "model objective": round(model_obj, 6),
        "true objective": round(sim["obj"], 6),
        "model - true": round(model_obj - sim["obj"], 6),
        "optimality gap %": round(100 * (sim["obj"] - true_ref) / abs(true_ref), 3),
        # V is mechanistic in every model, so V(tf) audits the U(t) rebuild.
        "V(tf) model": round(float(traj["V"][-1]), 4),
        "V(tf) sim": round(float(sim["V"][-1]), 4),
        "integrator": "ok" if sim["ok"] else "FAILED",
    }


# The mechanistic control, re-simulated, is the yardstick.
TRUE_REF = {ncp: simulate_true(REF_BY_NCP[ncp]["traj"]) for ncp in NCP_LIST}

play, SIMS = [], {}
for ncp in NCP_LIST:
    play.append(playback_row(ncp, "mechanistic", REF_BY_NCP[ncp]["obj"],
                             REF_BY_NCP[ncp]["traj"], TRUE_REF[ncp],
                             TRUE_REF[ncp]["obj"]))
    for fname, act in combos:
        tr = representative(ncp, fname, act)
        if tr is None:
            continue
        SIMS[(ncp, fname, act)] = sim = simulate_true(tr)
        sel = ok[(ok["ncp"] == ncp) & (ok["formulation"] == fname)
                 & (ok["activation"] == act)]
        play.append(playback_row(ncp, method_label(fname, act),
                                 float(sel["objective"].median()), tr, sim,
                                 TRUE_REF[ncp]["obj"]))

play_df = pd.DataFrame(play)
print("Feed profiles played back through the TRUE plant\n")
print("  model objective  = what the optimizer believed")
print("  true objective   = what that same U(t) actually delivers")
print("  optimality gap % = true objective vs the re-simulated mechanistic optimum "
      "(positive is worse)\n")
print(play_df.to_string(index=False))

_dv = (play_df["V(tf) sim"] - play_df["V(tf) model"]).abs()
print(f"\nV(tf) audit: model and simulation differ by at most {_dv.max():.2e} L"
      + ("  -> every row was fed the same substrate" if _dv.max() < 1e-3 else
         "  -> WARNING: the U(t) rebuild does not match the model; rows are "
         "NOT comparable"))
if (play_df["integrator"] != "ok").any():
    print("WARNING: solve_ivp failed on at least one profile; its true objective "
          "is not trustworthy.")

In [ ]:
# Model belief (dashed) vs the true plant under the same feed (solid), one row
# per method, at the finest grid, for the states that carry the surrogate error.
ncp_show = max(NCP_LIST)
shown = [(f, a) for f, a in combos if (ncp_show, f, a) in SIMS]

fig, axes = plt.subplots(len(shown), 3, figsize=(14, 2.6 * len(shown)),
                         sharex=True, squeeze=False)
for r, (fname, act) in enumerate(shown):
    meth = method_label(fname, act)
    c = method_style(meth)["color"]
    tr, sim = representative(ncp_show, fname, act), SIMS[(ncp_show, fname, act)]
    for col, nm in enumerate(["S", "X", "P"]):
        a = axes[r, col]
        a.plot(TRUE_REF[ncp_show]["t"], TRUE_REF[ncp_show][nm], "-", color="black",
               lw=1.6, label="mechanistic feed, true plant")
        a.plot(tr["t"], tr[nm], "--", lw=1.6, color=c, label="hybrid model")
        a.plot(sim["t"], sim[nm], "-", lw=1.6, alpha=0.85, color=c,
               label="same feed, true plant")
        if nm == "S":
            a.set_yscale("symlog", linthresh=1e-3)
        a.set_ylabel(ylabel(nm))
        if col == 0:
            a.set_title(meth, loc="left", fontsize=9)
axes[0, -1].legend(fontsize=7)
for a in axes[-1]:
    a.set_xlabel("time (h)")
fig.suptitle(f"What each optimizer believed (dashed) and what its feed does in the "
             f"true plant (solid), ncp={ncp_show}", y=1.0)
fig.tight_layout()
fig.savefig("CS3_naive_tutorial_model_vs_truth.png")
plt.show()

In [ ]:
RESULTS.to_csv(RESULTS_CSV, index=False)
play_df.to_csv(PLAYBACK_CSV, index=False)
print(f"wrote {RESULTS_CSV}  ({len(RESULTS)} rows)")
print(f"wrote {PLAYBACK_CSV}  ({len(play_df)} rows)")
for f in ("CS3_naive_tutorial_build_vs_solve.png",
          "CS3_naive_tutorial_control_trajectories.png",
          "CS3_naive_tutorial_state_trajectories.png",
          "CS3_naive_tutorial_model_vs_truth.png"):
    print(f"wrote {f}")
for act in NETS:
    print(f"network {act!r}: {WEIGHTS_FMT.format(act=act, n=N_HIDDEN)}, "
          f"{KERAS_FMT.format(act=act, n=N_HIDDEN)}")